# 🌐 ReliefNet — AI-Powered Disaster Management Platform
## Notebook 01: Dataset Understanding, Preprocessing, Integration & Feature Engineering

---

### 📌 Project Overview

**ReliefNet** is a production-grade AI platform for post-disaster logistics and resource allocation across India. It combines:

| Component | Purpose |
|---|---|
| 🚛 Trucks | Ground-level relief material delivery |
| 🚁 UAVs (Drones) | Aerial delivery to isolated/flooded zones |
| 📈 Forecasting | Predict upcoming disaster severity & impact |
| 🗺️ GIS Routing | Optimize road + aerial paths for delivery |
| 🧠 Explainable AI | Transparent allocation reasoning for operators |
| 👤 Human-in-the-Loop | Relief coordinator oversight and overrides |

---

### 🎯 Why This Notebook Exists

In disaster management, **data is scattered, inconsistent, and incomplete**:

- EM-DAT tracks global disasters but uses inconsistent district spellings
- OpenStreetMap has rich GIS data but needs graph processing
- Data.gov.in has health and warehouse data in mismatched formats
- Kaggle datasets are research-grade but need real-world alignment

This notebook creates a **single unified, clean, ML-ready dataset** from all these sources — the foundation for all downstream forecasting, optimization, and reinforcement learning modules.

---

### 📦 Expected Output

1. **Cleaned + integrated district-level feature matrix**
2. **Vulnerability index per district**
3. **Road network graph with failure probability**
4. **Warehouse & shelter accessibility scores**
5. **MongoDB-ready collection pipeline**
6. **CSV + Parquet export for ML pipelines**

---

> ⚠️ **Note:** This notebook does NOT train any model. It prepares the highest-quality data foundation for all downstream AI components.

---
# Section 1 — Environment Setup

### Why this matters
Disaster-management data engineering requires a specialized stack:
- **GeoPandas / Shapely / OSMnx**: Geospatial data manipulation — critical for routing and region analysis
- **NetworkX**: Road network graph analysis — find isolated regions, shortest paths
- **RapidFuzz**: Fast fuzzy string matching — needed to reconcile inconsistent district names across datasets
- **PyMongo**: MongoDB integration for production-scale storage
- **Missingno**: Visual audit of missing data patterns — essential before any imputation

All libraries are installed with version pinning to ensure reproducibility in team pipelines.

In [ ]:
# ============================================================
# CELL 1.1 — Install Dependencies
# Run this cell ONCE to install all required packages.
# In production, use a requirements.txt or conda environment.
# ============================================================

import subprocess, sys

packages = [
    'pandas', 'numpy', 'geopandas', 'networkx',
    'scikit-learn', 'pymongo', 'matplotlib', 'seaborn',
    'missingno', 'rapidfuzz', 'pyproj', 'shapely',
    'osmnx', 'folium', 'plotly', 'openpyxl',
    'pyarrow', 'fastparquet', 'tqdm', 'colorlog'
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])

print('✅ All packages installed successfully.')

In [ ]:
# ============================================================
# CELL 1.2 — Core Imports
# ============================================================

# --- Standard Library ---
import os
import re
import json
import logging
import warnings
from pathlib import Path
from datetime import datetime
from typing import Optional, Union, Dict, List, Tuple, Any

# --- Data Manipulation ---
import numpy as np
import pandas as pd

# --- Geospatial ---
import geopandas as gpd
from shapely.geometry import Point, LineString, Polygon
from pyproj import CRS, Transformer

# --- Graph / Network ---
import networkx as nx

# --- Machine Learning ---
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer, KNNImputer

# --- Fuzzy Matching ---
from rapidfuzz import fuzz, process as rfuzz_process

# --- Database ---
from pymongo import MongoClient, ASCENDING, GEOSPHERE
from pymongo.errors import BulkWriteError, DuplicateKeyError

# --- Visualization ---
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import missingno as msno
import folium
from folium.plugins import HeatMap
import plotly.express as px
import plotly.graph_objects as go

# --- Progress ---
from tqdm import tqdm

# --- Suppress noisy warnings in notebook ---
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

print('✅ All imports successful.')
print(f'   pandas   : {pd.__version__}')
print(f'   numpy    : {np.__version__}')
print(f'   geopandas: {gpd.__version__}')

In [ ]:
# ============================================================
# CELL 1.3 — Logging Configuration
# Production pipelines need traceable logs.
# We set up a dual handler: console + file log.
# ============================================================

def setup_logger(name: str = 'reliefnet', log_file: str = 'reliefnet_pipeline.log') -> logging.Logger:
    """
    Configure and return a production-grade logger.
    Logs to both console and file.
    """
    logger = logging.getLogger(name)
    logger.setLevel(logging.DEBUG)

    fmt = logging.Formatter(
        '[%(asctime)s] %(levelname)s — %(name)s — %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )

    # Console handler
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)

    # File handler
    fh = logging.FileHandler(log_file)
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)

    if not logger.handlers:
        logger.addHandler(ch)
        logger.addHandler(fh)

    return logger


logger = setup_logger()
logger.info('ReliefNet pipeline initialized.')

In [ ]:
# ============================================================
# CELL 1.4 — Path Configuration
# Never hardcode paths. Use pathlib.Path for portability.
# Structure:
#   data/raw/        — original untouched datasets
#   data/processed/  — cleaned intermediate files
#   data/exports/    — final ML-ready outputs
#   reports/         — EDA reports, validation summaries
# ============================================================

BASE_DIR      = Path().resolve()
RAW_DIR       = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
EXPORT_DIR    = BASE_DIR / 'data' / 'exports'
REPORT_DIR    = BASE_DIR / 'reports'
GIS_DIR       = RAW_DIR / 'gis'
EMDAT_DIR     = RAW_DIR / 'emdat'
GOVDATA_DIR   = RAW_DIR / 'govdata'
KAGGLE_DIR    = RAW_DIR / 'kaggle'

# Create directories if they don't exist
for d in [RAW_DIR, PROCESSED_DIR, EXPORT_DIR, REPORT_DIR, GIS_DIR,
          EMDAT_DIR, GOVDATA_DIR, KAGGLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

logger.info(f'Base directory: {BASE_DIR}')
print(f'✅ Directory structure ready under: {BASE_DIR}')

---
# Section 2 — Dataset Loader Framework

### Why a reusable loader framework?

In disaster management projects, data arrives in multiple formats from multiple agencies:
- UN agencies send **CSV** with UTF-8 or ISO-8859-1 encoding
- Government portals send **Excel** files with merged headers
- GIS systems output **Shapefiles** or **GeoJSON**
- APIs return **JSON**

A unified loader framework:
1. Handles all formats with a single function call
2. Auto-detects encoding to prevent `UnicodeDecodeError` crashes
3. Logs every load operation for audit trails
4. Returns consistent pandas/geopandas objects for downstream processing

In [ ]:
# ============================================================
# CELL 2.1 — Dataset Loader Functions
# ============================================================

def load_csv(
    path: Union[str, Path],
    encoding: str = 'utf-8',
    low_memory: bool = False,
    **kwargs
) -> pd.DataFrame:
    """
    Load a CSV file with automatic encoding fallback.

    Args:
        path: File path to the CSV.
        encoding: Primary encoding to attempt (default utf-8).
        low_memory: If True, pandas uses chunked inference (faster for large files).
        **kwargs: Additional arguments forwarded to pd.read_csv.

    Returns:
        pd.DataFrame with loaded data.

    Raises:
        FileNotFoundError: If file does not exist.
        ValueError: If file cannot be parsed.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'CSV not found: {path}')

    fallback_encodings = [encoding, 'utf-8-sig', 'latin-1', 'ISO-8859-1', 'cp1252']

    for enc in fallback_encodings:
        try:
            df = pd.read_csv(path, encoding=enc, low_memory=low_memory, **kwargs)
            logger.info(f'[load_csv] Loaded {path.name} — shape={df.shape}, encoding={enc}')
            return df
        except (UnicodeDecodeError, pd.errors.ParserError):
            continue

    raise ValueError(f'Cannot parse CSV with any known encoding: {path}')


def load_excel(
    path: Union[str, Path],
    sheet_name: Union[str, int] = 0,
    header: int = 0,
    **kwargs
) -> pd.DataFrame:
    """
    Load an Excel file (.xlsx / .xls).

    Args:
        path: File path.
        sheet_name: Sheet name or index (default: first sheet).
        header: Row number(s) to use as column names.
        **kwargs: Additional arguments for pd.read_excel.

    Returns:
        pd.DataFrame.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Excel file not found: {path}')

    try:
        df = pd.read_excel(path, sheet_name=sheet_name, header=header, **kwargs)
        logger.info(f'[load_excel] Loaded {path.name} — shape={df.shape}, sheet={sheet_name}')
        return df
    except Exception as e:
        raise ValueError(f'Excel load error [{path}]: {e}')


def load_json(
    path: Union[str, Path],
    orient: Optional[str] = None,
    **kwargs
) -> pd.DataFrame:
    """
    Load a JSON file and return as DataFrame.

    Args:
        path: File path.
        orient: JSON structure orientation (records, split, index, etc).
        **kwargs: Additional arguments for pd.read_json.

    Returns:
        pd.DataFrame.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'JSON not found: {path}')

    try:
        df = pd.read_json(path, orient=orient, **kwargs)
        logger.info(f'[load_json] Loaded {path.name} — shape={df.shape}')
        return df
    except Exception as e:
        raise ValueError(f'JSON load error [{path}]: {e}')


def load_geojson(
    path: Union[str, Path],
    crs: str = 'EPSG:4326'
) -> gpd.GeoDataFrame:
    """
    Load a GeoJSON file as a GeoDataFrame.

    Args:
        path: File path to GeoJSON.
        crs: Coordinate Reference System (default WGS84).

    Returns:
        gpd.GeoDataFrame.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'GeoJSON not found: {path}')

    try:
        gdf = gpd.read_file(str(path))
        if gdf.crs is None:
            gdf = gdf.set_crs(crs)
        elif gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs('EPSG:4326')
        logger.info(f'[load_geojson] Loaded {path.name} — {len(gdf)} features, CRS={gdf.crs}')
        return gdf
    except Exception as e:
        raise ValueError(f'GeoJSON load error [{path}]: {e}')


def load_shapefile(
    path: Union[str, Path],
    crs: str = 'EPSG:4326'
) -> gpd.GeoDataFrame:
    """
    Load an ESRI Shapefile (.shp) as a GeoDataFrame.
    Reprojects to WGS84 if needed (standard for lat/lon operations).

    Args:
        path: Path to the .shp file.
        crs: Target CRS (default WGS84 / EPSG:4326).

    Returns:
        gpd.GeoDataFrame.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Shapefile not found: {path}')

    try:
        gdf = gpd.read_file(str(path))
        if gdf.crs is None:
            gdf = gdf.set_crs(crs)
            logger.warning(f'[load_shapefile] CRS was None — set to {crs}')
        elif gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs('EPSG:4326')
            logger.info(f'[load_shapefile] Reprojected to EPSG:4326')
        logger.info(f'[load_shapefile] Loaded {path.name} — {len(gdf)} features')
        return gdf
    except Exception as e:
        raise ValueError(f'Shapefile load error [{path}]: {e}')


def smart_loader(
    path: Union[str, Path],
    **kwargs
) -> Union[pd.DataFrame, gpd.GeoDataFrame]:
    """
    Auto-detect file format and call the appropriate loader.
    Supports: .csv, .xlsx, .xls, .json, .geojson, .shp

    Args:
        path: Path to any supported file.
        **kwargs: Forwarded to the appropriate loader.

    Returns:
        DataFrame or GeoDataFrame.
    """
    ext = Path(path).suffix.lower()
    dispatch = {
        '.csv':     load_csv,
        '.xlsx':    load_excel,
        '.xls':     load_excel,
        '.json':    load_json,
        '.geojson': load_geojson,
        '.shp':     load_shapefile,
    }
    if ext not in dispatch:
        raise ValueError(f'Unsupported file extension: {ext}')
    return dispatch[ext](path, **kwargs)


print('✅ Dataset loader framework ready.')

---
# Section 3 — Dataset Discovery & Automated EDA

### Why automated EDA before any preprocessing?

Blind preprocessing leads to **information loss** and **biased features**. Before touching any data:

1. **Column profiling** — understand what each column means in a disaster context
2. **Null detection** — identify which districts or time periods have missing records
3. **Type detection** — are dates stored as strings? Are lat/lons stored as text?
4. **Geospatial column detection** — identify columns that can be converted to geometry
5. **Temporal column detection** — identify date/time fields for timeline construction
6. **Distribution analysis** — detect outliers (e.g., 999999 as null placeholders in government data)

These automated functions work on **any** dataset — EM-DAT, government data, or Kaggle files.

In [ ]:
# ============================================================
# CELL 3.1 — Automated Column Profiler
# ============================================================

def profile_dataset(
    df: pd.DataFrame,
    dataset_name: str = 'Dataset'
) -> pd.DataFrame:
    """
    Generate a comprehensive column-level profile report for any DataFrame.

    Detects:
    - Data type
    - Null count and percentage
    - Unique value count
    - Sample values
    - Whether column is likely geospatial (lat/lon)
    - Whether column is likely temporal (date/time)

    Args:
        df: Input DataFrame.
        dataset_name: Label for display.

    Returns:
        pd.DataFrame with column-level profile.
    """
    print(f'\n{'='*60}')
    print(f'  📋 Dataset Profile: {dataset_name}')
    print(f'{'='*60}')
    print(f'  Shape       : {df.shape[0]:,} rows × {df.shape[1]} columns')
    print(f'  Memory Usage: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB')
    print(f'  Duplicate Rows: {df.duplicated().sum():,}')
    print(f'{'='*60}\n')

    GEO_KEYWORDS  = {'lat', 'latitude', 'lon', 'longitude', 'lng', 'x', 'y',
                     'geometry', 'geom', 'coordinates', 'centroid'}
    TIME_KEYWORDS = {'date', 'time', 'year', 'month', 'day', 'period',
                     'start', 'end', 'timestamp', 'datetime', 'created', 'updated'}

    rows = []
    for col in df.columns:
        null_count   = df[col].isna().sum()
        null_pct     = (null_count / len(df)) * 100
        unique_count = df[col].nunique()
        dtype        = str(df[col].dtype)
        samples      = df[col].dropna().head(3).tolist()
        col_lower    = col.lower().replace(' ', '_')

        is_geo  = any(kw in col_lower for kw in GEO_KEYWORDS)
        is_time = (any(kw in col_lower for kw in TIME_KEYWORDS) or
                   pd.api.types.is_datetime64_any_dtype(df[col]))

        rows.append({
            'Column':        col,
            'DType':         dtype,
            'Null_Count':    null_count,
            'Null_%':        round(null_pct, 2),
            'Unique_Values': unique_count,
            'Is_Geospatial': '🌍' if is_geo  else '',
            'Is_Temporal':   '🕐' if is_time else '',
            'Sample_Values': str(samples)
        })

    profile_df = pd.DataFrame(rows)
    # Highlight high-null columns
    high_null = profile_df[profile_df['Null_%'] > 50]
    if not high_null.empty:
        print(f'  ⚠️  Columns with >50% nulls ({len(high_null)}): {list(high_null["Column"])}')

    return profile_df


def plot_missing_heatmap(
    df: pd.DataFrame,
    title: str = 'Missing Value Heatmap',
    figsize: Tuple[int, int] = (14, 6)
) -> None:
    """
    Render a missingno matrix + bar chart to visualize null patterns.

    WHY: In disaster datasets, missing values are rarely random —
    they often cluster by region (e.g., northeast India), year (pre-1990),
    or event type. Seeing the pattern informs the right imputation strategy.

    Args:
        df: Input DataFrame.
        title: Plot title.
        figsize: Figure size.
    """
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    fig.suptitle(title, fontsize=14, fontweight='bold')

    # Matrix — shows where nulls are located row-by-row
    msno.matrix(df, ax=axes[0], sparkline=False, fontsize=9)
    axes[0].set_title('Null Matrix (white = missing)', fontsize=10)

    # Bar — shows overall completeness per column
    msno.bar(df, ax=axes[1], fontsize=9, color='steelblue')
    axes[1].set_title('Column Completeness (%)', fontsize=10)

    plt.tight_layout()
    plt.savefig(REPORT_DIR / f'missing_{title.replace(" ","_")}.png', dpi=120, bbox_inches='tight')
    plt.show()


def plot_correlation_heatmap(
    df: pd.DataFrame,
    title: str = 'Correlation Heatmap',
    figsize: Tuple[int, int] = (12, 9)
) -> None:
    """
    Plot a correlation heatmap for numeric columns.

    WHY: Correlations reveal multi-collinearity before feature engineering.
    For example, flood_frequency and river_proximity are highly correlated
    — including both in a vulnerability index would double-count risk.

    Args:
        df: Input DataFrame.
        title: Plot title.
        figsize: Figure size.
    """
    numeric_df = df.select_dtypes(include=[np.number])
    if numeric_df.empty:
        print('  ⚠️  No numeric columns found for correlation plot.')
        return

    corr = numeric_df.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))

    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(
        corr, mask=mask, annot=True, fmt='.2f',
        cmap='RdYlGn', center=0, linewidths=0.5,
        annot_kws={'size': 8}, ax=ax
    )
    ax.set_title(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(REPORT_DIR / f'corr_{title.replace(" ","_")}.png', dpi=120, bbox_inches='tight')
    plt.show()


def plot_distributions(
    df: pd.DataFrame,
    title: str = 'Feature Distributions',
    max_cols: int = 12
) -> None:
    """
    Plot histograms for all numeric columns.

    WHY: Government disaster data often contains extreme outliers
    (e.g., 999999 used as null placeholder, or a single event with
    1 million affected vs typical 1000-100000). Distributions reveal these.

    Args:
        df: Input DataFrame.
        title: Figure title.
        max_cols: Maximum columns to plot.
    """
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()[:max_cols]
    if not num_cols:
        print('  ⚠️  No numeric columns to plot.')
        return

    n = len(num_cols)
    cols = min(4, n)
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3))
    axes = np.array(axes).flatten()

    for i, col in enumerate(num_cols):
        axes[i].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.85)
        axes[i].set_title(col, fontsize=9)
        axes[i].set_xlabel('')
        axes[i].grid(axis='y', alpha=0.3)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(REPORT_DIR / f'dist_{title.replace(" ","_")}.png', dpi=120, bbox_inches='tight')
    plt.show()


print('✅ EDA helper functions ready.')

In [ ]:
# ============================================================
# CELL 3.2 — EM-DAT Dataset Discovery
# USAGE: Place your EM-DAT CSV in data/raw/emdat/
# EM-DAT columns typically include:
#   DisNo, Year, Disaster Type, Country, Province, District,
#   Total Deaths, Total Affected, Total Damage (USD)
# ============================================================

# --- Load EM-DAT Disaster Data ---
emdat_files = list(EMDAT_DIR.glob('*.csv'))
if emdat_files:
    emdat_df = load_csv(emdat_files[0])

    # Normalize column names: lowercase, replace spaces with underscores
    emdat_df.columns = [
        re.sub(r'[^\w]', '_', c.strip().lower()) for c in emdat_df.columns
    ]

    # Run profile
    emdat_profile = profile_dataset(emdat_df, 'EM-DAT Disaster Dataset')
    display(emdat_profile)

    # Missing value visualization
    plot_missing_heatmap(emdat_df, 'EM-DAT Missing Values')

    # Distributions
    plot_distributions(emdat_df, 'EM-DAT Numeric Distributions')

    # Disaster type breakdown
    if 'disaster_type' in emdat_df.columns:
        fig = px.bar(
            emdat_df['disaster_type'].value_counts().reset_index(),
            x='disaster_type', y='count',
            title='EM-DAT: Disaster Type Distribution (India)',
            labels={'disaster_type': 'Type', 'count': 'Events'},
            color='disaster_type'
        )
        fig.update_layout(showlegend=False)
        fig.show()

else:
    print('⚠️  No EM-DAT file found in data/raw/emdat/.')
    print('    Creating synthetic EM-DAT sample for pipeline demonstration...')

    # --- Synthetic EM-DAT for pipeline demonstration ---
    np.random.seed(42)
    n = 500
    states = ['Maharashtra', 'Kerala', 'Odisha', 'Assam', 'Bihar',
              'West Bengal', 'Tamil Nadu', 'Andhra Pradesh', 'Gujarat', 'Rajasthan']
    districts_sample = [
        'Mumbai', 'Pune', 'Ernakulam', 'Thrissur', 'Puri', 'Khordha',
        'Kamrup', 'Nagaon', 'Patna', 'Muzaffarpur', 'Kolkata', 'Howrah',
        'Chennai', 'Madurai', 'Guntur', 'Krishna', 'Ahmedabad', 'Surat',
        'Jaipur', 'Jodhpur'
    ]
    disaster_types = ['Flood', 'Cyclone', 'Earthquake', 'Drought', 'Landslide']

    emdat_df = pd.DataFrame({
        'dis_no':            [f'IND-{2000+i//50}-{str(i).zfill(4)}' for i in range(n)],
        'year':              np.random.randint(2000, 2024, n),
        'start_month':       np.random.randint(1, 13, n),
        'end_month':         np.random.randint(1, 13, n),
        'disaster_type':     np.random.choice(disaster_types, n),
        'disaster_subtype':  np.random.choice(['Riverine', 'Flash', 'Coastal', 'N/A'], n),
        'country':           'India',
        'province':          np.random.choice(states, n),
        'district':          np.random.choice(districts_sample, n),
        'total_deaths':      np.random.choice([np.nan, 10, 25, 100, 500, 1200], n, p=[0.1,0.3,0.2,0.2,0.1,0.1]),
        'total_affected':    np.random.choice([np.nan, 5000, 50000, 200000, 1000000], n, p=[0.05,0.3,0.3,0.2,0.15]),
        'total_damage_usd':  np.random.choice([np.nan, 1e6, 5e6, 1e7, 5e7], n, p=[0.2,0.3,0.2,0.2,0.1]),
        'latitude':          np.random.uniform(8.0, 35.0, n),
        'longitude':         np.random.uniform(68.0, 97.0, n),
    })

    emdat_df.to_csv(EMDAT_DIR / 'emdat_india_synthetic.csv', index=False)
    logger.info(f'Synthetic EM-DAT created: {emdat_df.shape}')

    emdat_profile = profile_dataset(emdat_df, 'EM-DAT (Synthetic)')
    display(emdat_profile)
    plot_missing_heatmap(emdat_df, 'EM-DAT Missing Values')
    plot_distributions(emdat_df, 'EM-DAT Numeric Distributions')

In [ ]:
# ============================================================
# CELL 3.3 — Government & Infrastructure Dataset Discovery
# Handles NFHS-5, warehouse inventory, transport infrastructure
# ============================================================

# --- Attempt to load real government data ---
govdata_files = list(GOVDATA_DIR.glob('*.csv')) + list(GOVDATA_DIR.glob('*.xlsx'))

gov_datasets: Dict[str, pd.DataFrame] = {}

if govdata_files:
    for fpath in govdata_files:
        try:
            df = smart_loader(fpath)
            key = fpath.stem.lower().replace(' ', '_')
            gov_datasets[key] = df
            profile_dataset(df, fpath.name)
        except Exception as e:
            logger.warning(f'Could not load {fpath.name}: {e}')

else:
    print('⚠️  No government data files found. Creating synthetic infrastructure dataset...')

    # --- Synthetic Infrastructure + Health Indicators ---
    INDIAN_DISTRICTS = [
        ('Maharashtra', 'Mumbai'), ('Maharashtra', 'Pune'), ('Maharashtra', 'Nagpur'),
        ('Kerala', 'Ernakulam'), ('Kerala', 'Thrissur'), ('Kerala', 'Kozhikode'),
        ('Odisha', 'Puri'), ('Odisha', 'Khordha'), ('Odisha', 'Cuttack'),
        ('Assam', 'Kamrup'), ('Assam', 'Nagaon'), ('Assam', 'Barpeta'),
        ('Bihar', 'Patna'), ('Bihar', 'Muzaffarpur'), ('Bihar', 'Darbhanga'),
        ('West Bengal', 'Kolkata'), ('West Bengal', 'Howrah'), ('West Bengal', 'Midnapore'),
        ('Tamil Nadu', 'Chennai'), ('Tamil Nadu', 'Madurai'),
        ('Andhra Pradesh', 'Guntur'), ('Andhra Pradesh', 'Krishna'),
        ('Gujarat', 'Ahmedabad'), ('Gujarat', 'Surat'),
        ('Rajasthan', 'Jaipur'), ('Rajasthan', 'Jodhpur'),
        ('Uttarakhand', 'Dehradun'), ('Uttarakhand', 'Haridwar'),
        ('Himachal Pradesh', 'Shimla'), ('Himachal Pradesh', 'Kangra'),
        ('Jammu & Kashmir', 'Srinagar'), ('Jammu & Kashmir', 'Jammu'),
        ('Manipur', 'Imphal'), ('Nagaland', 'Kohima'), ('Tripura', 'Agartala')
    ]

    np.random.seed(0)
    n = len(INDIAN_DISTRICTS)

    infra_df = pd.DataFrame({
        'state':                  [x[0] for x in INDIAN_DISTRICTS],
        'district':               [x[1] for x in INDIAN_DISTRICTS],
        'population':             np.random.randint(100_000, 5_000_000, n),
        'area_sq_km':             np.random.randint(500, 15000, n),
        'num_hospitals':          np.random.randint(2, 80, n),
        'num_warehouses':         np.random.randint(0, 15, n),
        'num_shelters':           np.random.randint(5, 200, n),
        'shelter_capacity_total': np.random.randint(1000, 100000, n),
        'num_telecom_towers':     np.random.randint(10, 500, n),
        'road_length_km':         np.random.randint(100, 5000, n),
        'num_bridges':            np.random.randint(5, 300, n),
        'has_airport':            np.random.choice([0, 1], n, p=[0.7, 0.3]),
        'has_railway':            np.random.choice([0, 1], n, p=[0.4, 0.6]),
        'river_proximity_km':     np.random.uniform(0, 100, n),
        'coastal_district':       np.random.choice([0, 1], n, p=[0.7, 0.3]),
        'avg_elevation_m':        np.random.uniform(0, 3000, n),
        # NFHS-5 indicators
        'nfhs_stunting_pct':      np.random.uniform(10, 50, n),
        'nfhs_underweight_pct':   np.random.uniform(10, 45, n),
        'nfhs_anemia_children':   np.random.uniform(30, 90, n),
        'nfhs_institutional_birth': np.random.uniform(40, 99, n),
        # Warehouse inventory
        'warehouse_rice_tons':    np.random.uniform(0, 5000, n),
        'warehouse_wheat_tons':   np.random.uniform(0, 3000, n),
        'warehouse_medicine_kits': np.random.randint(0, 500, n),
        'warehouse_tarpaulin_units': np.random.randint(0, 2000, n),
    })

    # Introduce realistic missing values (10-15%)
    for col in ['num_telecom_towers', 'nfhs_stunting_pct', 'river_proximity_km',
                'warehouse_rice_tons', 'nfhs_anemia_children']:
        mask = np.random.rand(n) < 0.12
        infra_df.loc[mask, col] = np.nan

    infra_df.to_csv(GOVDATA_DIR / 'infrastructure_health_synthetic.csv', index=False)
    gov_datasets['infrastructure_health'] = infra_df
    logger.info(f'Synthetic infrastructure dataset created: {infra_df.shape}')

    infra_profile = profile_dataset(infra_df, 'Infrastructure & Health (Synthetic)')
    display(infra_profile)
    plot_missing_heatmap(infra_df, 'Infrastructure Missing Values')
    plot_correlation_heatmap(infra_df, 'Infrastructure Correlations')

---
# Section 4 — District Name Standardization

### 🚨 This is one of the most critical steps in Indian disaster data engineering.

India has **~750 districts** across 28 states and 8 UTs. Across datasets:
- EM-DAT may say **"Kamrup Metro"** while government data says **"Kamrup (M)"**
- Older datasets may use **"Bombay"** instead of **"Mumbai"**
- Kaggle datasets may use **"Vishakhapatnam"** instead of **"Visakhapatnam"**
- Some datasets use **Hindi transliterations** (e.g., "Patna" vs "Patnā")

Without standardization, a JOIN between two datasets will silently drop **hundreds of records** — corrupting vulnerability scores and routing decisions.

**Solution**: Fuzzy matching using RapidFuzz against a canonical district master table.
- Similarity score ≥ 85: auto-match
- Similarity score 60-84: flag for human review
- Similarity score < 60: mark as unmatched

In [ ]:
# ============================================================
# CELL 4.1 — District Master Table
# Source: LGD (Local Government Directory) of India
# ============================================================

# Canonical district master — ideally loaded from LGD official list
# This is an expanded representative sample; replace with full LGD list in production

DISTRICT_MASTER = [
    # Format: (State, Canonical District Name, LGD Code)
    ('Maharashtra',       'Mumbai',           'MH_001'),
    ('Maharashtra',       'Pune',             'MH_002'),
    ('Maharashtra',       'Nagpur',           'MH_003'),
    ('Maharashtra',       'Thane',            'MH_004'),
    ('Maharashtra',       'Nashik',           'MH_005'),
    ('Maharashtra',       'Aurangabad',       'MH_006'),
    ('Kerala',            'Ernakulam',        'KL_001'),
    ('Kerala',            'Thrissur',         'KL_002'),
    ('Kerala',            'Kozhikode',        'KL_003'),
    ('Kerala',            'Thiruvananthapuram','KL_004'),
    ('Kerala',            'Kollam',           'KL_005'),
    ('Odisha',            'Puri',             'OD_001'),
    ('Odisha',            'Khordha',          'OD_002'),
    ('Odisha',            'Cuttack',          'OD_003'),
    ('Odisha',            'Kendrapara',       'OD_004'),
    ('Assam',             'Kamrup',           'AS_001'),
    ('Assam',             'Kamrup Metropolitan','AS_002'),
    ('Assam',             'Nagaon',           'AS_003'),
    ('Assam',             'Barpeta',          'AS_004'),
    ('Assam',             'Dhubri',           'AS_005'),
    ('Bihar',             'Patna',            'BR_001'),
    ('Bihar',             'Muzaffarpur',      'BR_002'),
    ('Bihar',             'Darbhanga',        'BR_003'),
    ('Bihar',             'Bhagalpur',        'BR_004'),
    ('West Bengal',       'Kolkata',          'WB_001'),
    ('West Bengal',       'Howrah',           'WB_002'),
    ('West Bengal',       'Paschim Medinipur','WB_003'),
    ('West Bengal',       'Purba Medinipur',  'WB_004'),
    ('Tamil Nadu',        'Chennai',          'TN_001'),
    ('Tamil Nadu',        'Madurai',          'TN_002'),
    ('Tamil Nadu',        'Coimbatore',       'TN_003'),
    ('Andhra Pradesh',    'Guntur',           'AP_001'),
    ('Andhra Pradesh',    'Krishna',          'AP_002'),
    ('Andhra Pradesh',    'Visakhapatnam',    'AP_003'),
    ('Gujarat',           'Ahmedabad',        'GJ_001'),
    ('Gujarat',           'Surat',            'GJ_002'),
    ('Gujarat',           'Vadodara',         'GJ_003'),
    ('Rajasthan',         'Jaipur',           'RJ_001'),
    ('Rajasthan',         'Jodhpur',          'RJ_002'),
    ('Rajasthan',         'Barmer',           'RJ_003'),
    ('Uttarakhand',       'Dehradun',         'UK_001'),
    ('Uttarakhand',       'Haridwar',         'UK_002'),
    ('Uttarakhand',       'Chamoli',          'UK_003'),
    ('Himachal Pradesh',  'Shimla',           'HP_001'),
    ('Himachal Pradesh',  'Kangra',           'HP_002'),
    ('Himachal Pradesh',  'Mandi',            'HP_003'),
    ('Jammu and Kashmir', 'Srinagar',         'JK_001'),
    ('Jammu and Kashmir', 'Jammu',            'JK_002'),
    ('Manipur',           'Imphal East',      'MN_001'),
    ('Manipur',           'Imphal West',      'MN_002'),
    ('Nagaland',          'Kohima',           'NL_001'),
    ('Tripura',           'West Tripura',     'TR_001'),
    ('Tripura',           'Dhalai',           'TR_002'),
]

master_df = pd.DataFrame(
    DISTRICT_MASTER,
    columns=['state', 'district_canonical', 'lgd_code']
)

print(f'✅ District master loaded: {len(master_df)} canonical districts')
display(master_df.head(10))

In [ ]:
# ============================================================
# CELL 4.2 — Fuzzy District Matching Engine
# ============================================================

def normalize_district_name(name: str) -> str:
    """
    Apply basic normalization before fuzzy matching.
    - Lowercase
    - Remove punctuation / special chars
    - Strip whitespace
    - Expand common abbreviations

    Args:
        name: Raw district name string.

    Returns:
        Normalized string.
    """
    if pd.isna(name):
        return ''
    name = str(name).strip().lower()
    name = re.sub(r'[^a-z0-9\s]', ' ', name)
    name = re.sub(r'\s+', ' ', name).strip()

    # Common abbreviation expansions
    abbr_map = {
        r'\bm\.?\b': 'metropolitan',
        r'\bnorth\b': 'north',
        r'\bsouth\b': 'south',
        r'\beast\b':  'east',
        r'\bwest\b':  'west',
        r'bombay':    'mumbai',
        r'calcutta':  'kolkata',
        r'madras':    'chennai',
        r'vishakhapatnam': 'visakhapatnam',
        r'trivandrum': 'thiruvananthapuram',
    }
    for pattern, replacement in abbr_map.items():
        name = re.sub(pattern, replacement, name)

    return name


def match_districts(
    raw_names: pd.Series,
    master_df: pd.DataFrame,
    score_threshold_auto: int = 85,
    score_threshold_review: int = 60
) -> pd.DataFrame:
    """
    Match a series of raw district names against the canonical master table
    using RapidFuzz token_sort_ratio (handles word-order variation).

    Scoring:
    - Score >= score_threshold_auto:   AUTO-MATCHED (high confidence)
    - score_threshold_review <= score < auto: REVIEW_NEEDED
    - Score < score_threshold_review: UNMATCHED

    Args:
        raw_names: Series of district name strings to match.
        master_df: DataFrame with 'district_canonical' and 'lgd_code' columns.
        score_threshold_auto:   Minimum score for automatic match (default 85).
        score_threshold_review: Minimum score to flag for human review (default 60).

    Returns:
        pd.DataFrame with columns:
        [raw_name, matched_district, lgd_code, confidence, match_status]
    """
    canonical_list = master_df['district_canonical'].tolist()
    canonical_norm = [normalize_district_name(c) for c in canonical_list]

    results = []
    for raw in tqdm(raw_names, desc='Matching districts'):
        norm = normalize_district_name(raw)
        if not norm:
            results.append({
                'raw_name': raw, 'matched_district': None,
                'lgd_code': None, 'confidence': 0,
                'match_status': 'MISSING'
            })
            continue

        # RapidFuzz: find best match among canonical list
        match_result = rfuzz_process.extractOne(
            norm,
            canonical_norm,
            scorer=fuzz.token_sort_ratio
        )

        if match_result is None:
            status, matched, lgd, score = 'UNMATCHED', None, None, 0
        else:
            best_norm, score, idx = match_result
            matched = canonical_list[idx]
            lgd     = master_df.iloc[idx]['lgd_code']

            if score >= score_threshold_auto:
                status = 'AUTO_MATCHED'
            elif score >= score_threshold_review:
                status = 'REVIEW_NEEDED'
            else:
                status = 'UNMATCHED'
                matched, lgd = None, None

        results.append({
            'raw_name':        raw,
            'matched_district': matched,
            'lgd_code':        lgd,
            'confidence':      score,
            'match_status':    status
        })

    return pd.DataFrame(results)


# --- Run district matching on EM-DAT data ---
if 'district' in emdat_df.columns:
    unique_districts = emdat_df['district'].dropna().unique()
    district_mapping = match_districts(pd.Series(unique_districts), master_df)

    print('\n📊 District Matching Summary:')
    print(district_mapping['match_status'].value_counts().to_string())

    # Save mapping table
    district_mapping.to_csv(REPORT_DIR / 'district_mapping_report.csv', index=False)
    print(f'\n✅ District mapping saved to reports/district_mapping_report.csv')

    # Apply mapping back to emdat_df
    mapping_dict = dict(zip(
        district_mapping['raw_name'],
        district_mapping['matched_district']
    ))
    emdat_df['district_canonical'] = emdat_df['district'].map(mapping_dict)
    emdat_df['district_lgd_code']  = emdat_df['district'].map(
        dict(zip(district_mapping['raw_name'], district_mapping['lgd_code']))
    )

    display(district_mapping.head(15))

---
# Section 5 — Missing Value Handling

### Strategy Selection Rationale

Different missing value patterns require different strategies:

| Pattern | Strategy | Reason |
|---|---|---|
| Random missingness (<15%) | **Median imputation** | Robust to outliers — disasters create extreme values |
| State-level clustered missingness | **State-wise group median** | Districts in same state share similar infrastructure |
| Temporal gaps | **Linear interpolation** | Disaster frequency trends change gradually |
| Infrastructure columns | **KNN imputation** | Similar-geography districts have similar infrastructure |
| >60% missing | **Drop column** | Too sparse to reliably impute; creates noise in ML |

⚠️ **We never use mean imputation for disaster data** — a single catastrophic event (1 million affected) would inflate the mean and corrupt imputed values for routine floods.

In [ ]:
# ============================================================
# CELL 5.1 — Missing Value Audit Report Generator
# ============================================================

def generate_missing_audit(
    df: pd.DataFrame,
    dataset_name: str = 'Dataset'
) -> pd.DataFrame:
    """
    Generate a structured missing-value audit report.
    Categorizes columns into imputation strategy buckets.

    Args:
        df: Input DataFrame.
        dataset_name: Label for logging.

    Returns:
        pd.DataFrame with audit results per column.
    """
    rows = []
    for col in df.columns:
        null_pct = df[col].isna().mean() * 100
        dtype    = str(df[col].dtype)

        if null_pct == 0:
            strategy = 'NONE_REQUIRED'
        elif null_pct <= 15:
            strategy = 'MEDIAN_IMPUTE' if 'float' in dtype or 'int' in dtype else 'MODE_IMPUTE'
        elif null_pct <= 40:
            strategy = 'KNN_IMPUTE' if 'float' in dtype or 'int' in dtype else 'PROPAGATE_FROM_STATE'
        elif null_pct <= 60:
            strategy = 'INTERPOLATE_OR_FLAG'
        else:
            strategy = 'DROP_COLUMN'

        rows.append({
            'column':          col,
            'dtype':           dtype,
            'null_count':      df[col].isna().sum(),
            'null_pct':        round(null_pct, 2),
            'recommended_strategy': strategy
        })

    audit_df = pd.DataFrame(rows).sort_values('null_pct', ascending=False)
    audit_df.to_csv(REPORT_DIR / f'missing_audit_{dataset_name.replace(" ","_")}.csv', index=False)
    logger.info(f'Missing audit for {dataset_name}: {len(audit_df)} columns analyzed')
    return audit_df


# Run audit on infra dataset
infra_audit = generate_missing_audit(infra_df, 'Infrastructure')
display(infra_audit[infra_audit['null_count'] > 0])

In [ ]:
# ============================================================
# CELL 5.2 — Missing Value Imputation Pipeline
# ============================================================

def impute_numeric_columns(
    df: pd.DataFrame,
    group_col: Optional[str] = 'state',
    knn_neighbors: int = 5
) -> pd.DataFrame:
    """
    Apply a tiered imputation strategy to numeric columns.

    Tier 1: If null_pct <= 15 → group median by state
    Tier 2: If null_pct 15-40 → KNN imputation
    Tier 3: If null_pct > 60 → drop column
    Tier 4: Any remaining → global median fallback

    Args:
        df: Input DataFrame.
        group_col: Column to group by for state-wise imputation.
        knn_neighbors: Number of neighbors for KNN imputation.

    Returns:
        Imputed DataFrame with audit trail columns added.
    """
    df = df.copy()
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    # --- Drop columns with >60% missing ---
    drop_cols = [c for c in num_cols if df[c].isna().mean() > 0.60]
    if drop_cols:
        logger.warning(f'Dropping high-null columns (>60%): {drop_cols}')
        df = df.drop(columns=drop_cols)
        num_cols = [c for c in num_cols if c not in drop_cols]

    # --- Tier 1: State-wise group median (≤15% null) ---
    low_null = [c for c in num_cols if 0 < df[c].isna().mean() <= 0.15]
    if low_null and group_col and group_col in df.columns:
        for col in low_null:
            df[col] = df.groupby(group_col)[col].transform(
                lambda x: x.fillna(x.median())
            )
            # Fallback to global median if state has all nulls
            df[col] = df[col].fillna(df[col].median())
        logger.info(f'[Tier 1] State-wise median imputed: {low_null}')

    # --- Tier 2: KNN imputation (15-60% null) ---
    mid_null = [c for c in num_cols if 0.15 < df[c].isna().mean() <= 0.60]
    if mid_null:
        knn_imputer = KNNImputer(n_neighbors=knn_neighbors)
        df[mid_null] = knn_imputer.fit_transform(df[mid_null])
        logger.info(f'[Tier 2] KNN imputed: {mid_null}')

    # --- Tier 4: Global median fallback for any remaining ---
    still_null = [c for c in num_cols if df[c].isna().any()]
    if still_null:
        for col in still_null:
            df[col] = df[col].fillna(df[col].median())
        logger.info(f'[Tier 4] Global median fallback: {still_null}')

    return df


# --- Apply imputation to infrastructure data ---
infra_df = impute_numeric_columns(infra_df, group_col='state')

# Verify
remaining_nulls = infra_df.isnull().sum().sum()
print(f'✅ Imputation complete. Remaining nulls: {remaining_nulls}')

---
# Section 6 — Temporal Data Cleaning

### Why temporal processing is non-trivial in disaster data

Disaster datasets mix date formats:
- EM-DAT uses **`YYYY-MM-DD`** for modern records, **`YYYY`** for historical
- Government data uses **`DD/MM/YYYY`** and **`DD-Mon-YYYY`** interchangeably
- Some datasets record **start date** and **end date** separately
- Some only have **year** (for historical events)

We need:
1. ISO-8601 normalized dates for all events
2. **Disaster duration** (end - start) in days → affects relief timeline
3. **Season indicator** (monsoon/winter/summer) → critical for UAV deployment planning
4. **Recency weight** → recent disasters more relevant for forecasting

In [ ]:
# ============================================================
# CELL 6.1 — Temporal Cleaning & Feature Creation
# ============================================================

def clean_temporal_columns(
    df: pd.DataFrame,
    year_col: str = 'year',
    month_col: Optional[str] = 'start_month',
    end_month_col: Optional[str] = 'end_month'
) -> pd.DataFrame:
    """
    Normalize temporal columns and derive disaster timeline features.

    Creates:
    - disaster_start_date (ISO-8601)
    - disaster_end_date   (ISO-8601)
    - disaster_duration_days
    - season (Monsoon / Winter / Pre-Monsoon / Post-Monsoon)
    - recency_weight (exponential decay from current year)
    - decade (for historical trend analysis)

    Args:
        df: Input DataFrame with disaster events.
        year_col: Name of year column.
        month_col: Name of start month column (1-12).
        end_month_col: Name of end month column.

    Returns:
        DataFrame with temporal features added.
    """
    df = df.copy()
    current_year = datetime.now().year

    # --- Convert year to int ---
    if year_col in df.columns:
        df[year_col] = pd.to_numeric(df[year_col], errors='coerce')

    # --- Build ISO start date ---
    if year_col in df.columns and month_col in df.columns:
        df['disaster_start_date'] = pd.to_datetime(
            df[year_col].astype(str) + '-' +
            df[month_col].fillna(1).astype(int).astype(str).str.zfill(2) + '-01',
            format='%Y-%m-%d', errors='coerce'
        )
    elif year_col in df.columns:
        df['disaster_start_date'] = pd.to_datetime(
            df[year_col].astype(str) + '-01-01', format='%Y-%m-%d', errors='coerce'
        )

    # --- Build ISO end date ---
    if year_col in df.columns and end_month_col in df.columns:
        df['disaster_end_date'] = pd.to_datetime(
            df[year_col].astype(str) + '-' +
            df[end_month_col].fillna(df[month_col] if month_col else 1).astype(int).astype(str).str.zfill(2) + '-01',
            format='%Y-%m-%d', errors='coerce'
        )

    # --- Disaster duration in days ---
    if 'disaster_start_date' in df.columns and 'disaster_end_date' in df.columns:
        df['disaster_duration_days'] = (
            df['disaster_end_date'] - df['disaster_start_date']
        ).dt.days.clip(lower=1)

    # --- Season indicator (Indian meteorological seasons) ---
    if month_col in df.columns:
        def get_season(month: Optional[float]) -> str:
            if pd.isna(month):
                return 'Unknown'
            m = int(month)
            if m in [6, 7, 8, 9]:   return 'Monsoon'      # Peak flood/cyclone season
            if m in [10, 11]:        return 'Post-Monsoon'  # Cyclone season (Bay of Bengal)
            if m in [12, 1, 2]:      return 'Winter'        # Cold wave, fog disrupts transport
            return 'Pre-Monsoon'                            # Heat waves, dry spells

        df['season'] = df[month_col].apply(get_season)

    # --- Decade for historical trend grouping ---
    if year_col in df.columns:
        df['decade'] = (df[year_col] // 10 * 10).astype('Int64')

    # --- Recency weight: recent disasters get higher weight in forecasting ---
    # Uses exponential decay: w = exp(-lambda * years_ago)
    # Lambda = 0.1 means ~10% importance decay per year
    if year_col in df.columns:
        years_ago = current_year - df[year_col]
        df['recency_weight'] = np.exp(-0.1 * years_ago.clip(lower=0))

    logger.info(f'Temporal features created: disaster_start_date, disaster_end_date, '
                f'disaster_duration_days, season, decade, recency_weight')
    return df


# --- Apply to EM-DAT ---
emdat_df = clean_temporal_columns(emdat_df, year_col='year',
                                   month_col='start_month', end_month_col='end_month')

print('✅ Temporal columns created:')
temporal_cols = ['disaster_start_date', 'disaster_end_date',
                 'disaster_duration_days', 'season', 'decade', 'recency_weight']
display(emdat_df[temporal_cols].head(10))

# --- Visualize seasonal disaster distribution ---
if 'season' in emdat_df.columns and 'disaster_type' in emdat_df.columns:
    season_counts = emdat_df.groupby(['season', 'disaster_type']).size().reset_index(name='count')
    fig = px.bar(
        season_counts, x='season', y='count', color='disaster_type',
        title='Disaster Frequency by Season and Type',
        barmode='stack',
        labels={'count': 'Number of Events', 'season': 'Season'}
    )
    fig.show()

---
# Section 7 — Geospatial Processing

### Why geospatial processing is the backbone of disaster logistics

Every relief decision is inherently spatial:
- **Which warehouse is closest** to the disaster zone? (Euclidean vs road distance)
- **Which roads are still accessible** when a bridge is flooded?
- **Which districts are isolated** due to broken road connectivity?
- **Where should UAVs launch from** given flight radius constraints?

This section:
1. Creates **district centroids** (geometric center of each district polygon)
2. Builds a **spatial index** for fast nearest-neighbor queries
3. Creates **infrastructure GeoDataFrames** (warehouses, hospitals, shelters)
4. Computes **proximity features** using spatial joins
5. Generates **flood region overlays** from EM-DAT coordinates

In [ ]:
# ============================================================
# CELL 7.1 — Create Infrastructure GeoDataFrames
# ============================================================

def create_infrastructure_geodataframe(
    df: pd.DataFrame,
    lat_col: str = 'latitude',
    lon_col: str = 'longitude',
    crs: str = 'EPSG:4326'
) -> gpd.GeoDataFrame:
    """
    Convert a DataFrame with lat/lon columns into a GeoDataFrame.

    Args:
        df: DataFrame with coordinate columns.
        lat_col: Latitude column name.
        lon_col: Longitude column name.
        crs: Coordinate reference system.

    Returns:
        gpd.GeoDataFrame with Point geometry.
    """
    df = df.copy()

    # Validate coordinate ranges for India
    if lat_col in df.columns and lon_col in df.columns:
        invalid_lat = ~df[lat_col].between(6.0, 38.0)
        invalid_lon = ~df[lon_col].between(67.0, 99.0)
        n_invalid = (invalid_lat | invalid_lon).sum()
        if n_invalid > 0:
            logger.warning(f'{n_invalid} records have coordinates outside India bounding box')

        geometry = [
            Point(lon, lat) if pd.notna(lon) and pd.notna(lat) else None
            for lat, lon in zip(df[lat_col], df[lon_col])
        ]
        gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=crs)
    else:
        raise ValueError(f'Columns {lat_col} or {lon_col} not found in DataFrame')

    logger.info(f'GeoDataFrame created: {len(gdf)} features, CRS={crs}')
    return gdf


def generate_synthetic_infrastructure_coords(
    infra_df: pd.DataFrame
) -> gpd.GeoDataFrame:
    """
    Assign approximate centroid coordinates to districts
    using a hard-coded lookup table for known districts.
    In production, replace with official LGD/GADM centroids.

    Args:
        infra_df: Infrastructure DataFrame with 'district' column.

    Returns:
        GeoDataFrame with geometry column.
    """
    # Approximate centroids for known districts
    DISTRICT_COORDS = {
        'Mumbai':       (18.9388, 72.8354),
        'Pune':         (18.5204, 73.8567),
        'Nagpur':       (21.1458, 79.0882),
        'Ernakulam':    (9.9816, 76.2999),
        'Thrissur':     (10.5276, 76.2144),
        'Kozhikode':    (11.2588, 75.7804),
        'Puri':         (19.8135, 85.8312),
        'Khordha':      (20.1823, 85.6953),
        'Cuttack':      (20.4625, 85.8830),
        'Kamrup':       (26.1445, 91.7362),
        'Nagaon':       (26.3466, 92.6844),
        'Barpeta':      (26.3219, 91.0036),
        'Patna':        (25.5941, 85.1376),
        'Muzaffarpur':  (26.1209, 85.3647),
        'Darbhanga':    (26.1542, 85.8918),
        'Kolkata':      (22.5726, 88.3639),
        'Howrah':       (22.5958, 88.2636),
        'Midnapore':    (22.4250, 87.3194),
        'Chennai':      (13.0827, 80.2707),
        'Madurai':      (9.9252, 78.1198),
        'Guntur':       (16.3067, 80.4365),
        'Krishna':      (16.5060, 80.6480),
        'Visakhapatnam':(17.6868, 83.2185),
        'Ahmedabad':    (23.0225, 72.5714),
        'Surat':        (21.1702, 72.8311),
        'Vadodara':     (22.3072, 73.1812),
        'Jaipur':       (26.9124, 75.7873),
        'Jodhpur':      (26.2389, 73.0243),
        'Barmer':       (25.7462, 71.3985),
        'Dehradun':     (30.3165, 78.0322),
        'Haridwar':     (29.9457, 78.1642),
        'Chamoli':      (30.3847, 79.4285),
        'Shimla':       (31.1048, 77.1734),
        'Kangra':       (32.0999, 76.2691),
        'Mandi':        (31.7103, 76.9320),
        'Srinagar':     (34.0837, 74.7973),
        'Jammu':        (32.7266, 74.8570),
        'Imphal':       (24.8170, 93.9368),
        'Kohima':       (25.6751, 94.1086),
        'Agartala':     (23.8315, 91.2868),
    }

    infra_df = infra_df.copy()
    infra_df['latitude']  = infra_df['district'].map(
        lambda d: DISTRICT_COORDS.get(d, (np.nan, np.nan))[0]
    )
    infra_df['longitude'] = infra_df['district'].map(
        lambda d: DISTRICT_COORDS.get(d, (np.nan, np.nan))[1]
    )

    # For unmatched districts, assign jittered state centroid
    unmatched = infra_df['latitude'].isna()
    if unmatched.any():
        logger.warning(f'{unmatched.sum()} districts have no coordinate lookup — using random assignment')
        infra_df.loc[unmatched, 'latitude']  = np.random.uniform(10, 30, unmatched.sum())
        infra_df.loc[unmatched, 'longitude'] = np.random.uniform(70, 95, unmatched.sum())

    return create_infrastructure_geodataframe(infra_df)


# --- Create GeoDataFrame ---
infra_gdf = generate_synthetic_infrastructure_coords(infra_df)
print(f'✅ Infrastructure GeoDataFrame: {len(infra_gdf)} districts')
display(infra_gdf[['district', 'state', 'latitude', 'longitude', 'geometry']].head(10))

In [ ]:
# ============================================================
# CELL 7.2 — Spatial Proximity Features
# Nearest warehouse, hospital, shelter per district centroid
# ============================================================

def compute_nearest_warehouse_distance(
    district_gdf: gpd.GeoDataFrame,
    warehouse_gdf: Optional[gpd.GeoDataFrame] = None
) -> pd.Series:
    """
    Calculate Euclidean distance (km) from each district centroid
    to the nearest warehouse.

    NOTE: Euclidean distance is used here for feature engineering.
    Section 8 (Road Network) computes actual road-distance.

    In disaster contexts, warehouse-district distance determines:
    - Truck delivery time estimate
    - Whether UAV delivery is preferable (no road access)
    - Priority for pre-positioning supplies before events

    Args:
        district_gdf: GeoDataFrame with district Point geometries.
        warehouse_gdf: Optional separate GeoDataFrame of warehouse locations.
                       If None, uses districts that have warehouses.

    Returns:
        pd.Series of distances in km.
    """
    # Project to metric CRS for accurate distance in km
    districts_m = district_gdf.to_crs('EPSG:32643')  # WGS 84 / UTM zone 43N (covers India)

    if warehouse_gdf is not None:
        warehouses_m = warehouse_gdf.to_crs('EPSG:32643')
    else:
        # Districts with warehouses serve as warehouse locations
        mask = district_gdf['num_warehouses'] > 0 if 'num_warehouses' in district_gdf.columns else district_gdf.index >= 0
        warehouses_m = districts_m[mask]

    if warehouses_m.empty:
        return pd.Series([np.nan] * len(districts_m), index=districts_m.index)

    distances = []
    for geom in districts_m.geometry:
        if geom is None or geom.is_empty:
            distances.append(np.nan)
            continue
        dist_m = warehouses_m.geometry.distance(geom).min()
        distances.append(dist_m / 1000.0)  # Convert to km

    return pd.Series(distances, index=district_gdf.index)


# --- Compute nearest warehouse distance ---
infra_gdf['nearest_warehouse_km'] = compute_nearest_warehouse_distance(infra_gdf)
print(f'Nearest warehouse distance (km):')
print(infra_gdf['nearest_warehouse_km'].describe().round(2))

In [ ]:
# ============================================================
# CELL 7.3 — Folium Interactive Map: District Infrastructure
# ============================================================

def create_infrastructure_map(
    gdf: gpd.GeoDataFrame,
    disaster_df: Optional[pd.DataFrame] = None,
    output_path: Optional[Path] = None
) -> folium.Map:
    """
    Create an interactive Folium map showing:
    - District centroids (color-coded by warehouse count)
    - Disaster event heatmap (from EM-DAT)
    - Hospital markers

    Args:
        gdf: Infrastructure GeoDataFrame.
        disaster_df: Optional EM-DAT DataFrame for heat overlay.
        output_path: Optional path to save HTML.

    Returns:
        folium.Map object.
    """
    # Center on India
    m = folium.Map(location=[20.5937, 78.9629], zoom_start=5,
                   tiles='CartoDB positron')

    # --- District markers ---
    for _, row in gdf.iterrows():
        if row.geometry is None:
            continue
        lat, lon = row.geometry.y, row.geometry.x
        warehouses = row.get('num_warehouses', 0)
        hospitals  = row.get('num_hospitals', 0)
        color = 'red' if warehouses == 0 else ('orange' if warehouses <= 3 else 'green')

        folium.CircleMarker(
            location=[lat, lon],
            radius=6,
            color=color,
            fill=True, fill_opacity=0.7,
            popup=folium.Popup(
                f"<b>{row.get('district','N/A')}</b><br>"
                f"State: {row.get('state','N/A')}<br>"
                f"Warehouses: {warehouses}<br>"
                f"Hospitals: {hospitals}<br>"
                f"Population: {row.get('population',0):,}",
                max_width=200
            ),
            tooltip=row.get('district', '')
        ).add_to(m)

    # --- Disaster heatmap overlay ---
    if disaster_df is not None and 'latitude' in disaster_df.columns:
        heat_data = disaster_df[['latitude', 'longitude']].dropna().values.tolist()
        if heat_data:
            HeatMap(heat_data, radius=15, blur=20, name='Disaster Density').add_to(m)

    # Legend
    legend_html = '''
    <div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
                background: white; padding: 10px; border-radius: 8px;
                border: 1px solid #ccc; font-size: 12px;">
    <b>Warehouse Status</b><br>
    <span style="color:red">●</span> No warehouses<br>
    <span style="color:orange">●</span> 1-3 warehouses<br>
    <span style="color:green">●</span> 4+ warehouses
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))

    if output_path:
        m.save(str(output_path))
        logger.info(f'Map saved: {output_path}')

    return m


infra_map = create_infrastructure_map(
    infra_gdf, disaster_df=emdat_df,
    output_path=REPORT_DIR / 'infrastructure_map.html'
)
infra_map

---
# Section 8 — Road Network Processing

### Why road network analysis matters for ReliefNet

A disaster-resilient logistics system must understand:
1. **Road connectivity** — How many paths exist between a warehouse and a district?
2. **Shortest path** — What is the fastest ground route?
3. **Vulnerability** — Which roads are at risk of flooding/landslide?
4. **Disconnected districts** — When roads fail, which districts are isolated?

This section uses **NetworkX** to build a road graph and compute these features. In production, feed real road data from OpenStreetMap via `osmnx`.

In [ ]:
# ============================================================
# CELL 8.1 — Road Network Graph Builder
# ============================================================

def build_district_road_graph(
    infra_gdf: gpd.GeoDataFrame,
    max_connection_km: float = 150.0
) -> nx.Graph:
    """
    Build a weighted undirected road graph where:
    - Nodes = districts (identified by LGD code or district name)
    - Edges = road connections between adjacent/nearby districts
    - Edge weight = Euclidean distance in km (proxy for road distance)
    - Edge attribute 'flood_risk' = based on elevation and river proximity

    In production: replace with OSMnx real road data.

    Args:
        infra_gdf: Infrastructure GeoDataFrame with district centroids.
        max_connection_km: Maximum distance to create an edge (default 150 km).

    Returns:
        nx.Graph with node and edge attributes.
    """
    G = nx.Graph()
    districts_m = infra_gdf.to_crs('EPSG:32643').copy()

    # --- Add nodes ---
    for idx, row in infra_gdf.iterrows():
        district_id = row.get('district', str(idx))
        G.add_node(district_id, **{
            'state':             row.get('state', 'Unknown'),
            'population':        row.get('population', 0),
            'num_warehouses':    row.get('num_warehouses', 0),
            'num_hospitals':     row.get('num_hospitals', 0),
            'num_shelters':      row.get('num_shelters', 0),
            'avg_elevation_m':   row.get('avg_elevation_m', 100),
            'river_proximity_km': row.get('river_proximity_km', 50),
            'lat':               row.geometry.y if row.geometry else 0,
            'lon':               row.geometry.x if row.geometry else 0,
        })

    # --- Add edges based on proximity ---
    district_list = list(infra_gdf['district'])
    geom_list     = list(districts_m.geometry)

    for i in range(len(district_list)):
        for j in range(i + 1, len(district_list)):
            if geom_list[i] is None or geom_list[j] is None:
                continue
            dist_m  = geom_list[i].distance(geom_list[j])
            dist_km = dist_m / 1000.0

            if dist_km <= max_connection_km:
                # Compute road failure probability based on flood/landslide risk
                elev_i = infra_gdf.iloc[i].get('avg_elevation_m', 100)
                elev_j = infra_gdf.iloc[j].get('avg_elevation_m', 100)
                riv_i  = infra_gdf.iloc[i].get('river_proximity_km', 50)
                riv_j  = infra_gdf.iloc[j].get('river_proximity_km', 50)

                # Higher flood risk: low elevation + close to river
                flood_risk = max(0, min(1,
                    0.4 * (1 / (1 + min(elev_i, elev_j) / 100)) +
                    0.6 * (1 / (1 + min(riv_i, riv_j) / 20))
                ))

                # Landslide risk: high elevation difference
                elev_diff = abs(elev_i - elev_j)
                landslide_risk = min(1, elev_diff / 1000)

                failure_prob = min(1, flood_risk * 0.7 + landslide_risk * 0.3)

                G.add_edge(
                    district_list[i], district_list[j],
                    weight=dist_km,
                    distance_km=dist_km,
                    flood_risk=round(flood_risk, 4),
                    landslide_risk=round(landslide_risk, 4),
                    failure_probability=round(failure_prob, 4)
                )

    logger.info(f'Road graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
    return G


# --- Build graph ---
road_graph = build_district_road_graph(infra_gdf, max_connection_km=150)
print(f'\n✅ Road Network Graph:')
print(f'   Nodes (districts): {road_graph.number_of_nodes()}')
print(f'   Edges (roads):     {road_graph.number_of_edges()}')
print(f'   Connected components: {nx.number_connected_components(road_graph)}')
print(f'   Average degree:    {sum(dict(road_graph.degree()).values()) / road_graph.number_of_nodes():.2f}')

In [ ]:
# ============================================================
# CELL 8.2 — Road Network Feature Extraction
# ============================================================

def extract_network_features(
    G: nx.Graph,
    district_list: List[str]
) -> pd.DataFrame:
    """
    Extract graph-theoretic features per district for ML.

    Features:
    - degree:              Number of road connections (higher = better connected)
    - betweenness_centrality: How critical this node is for routing
    - avg_road_failure_prob:  Mean failure probability of adjacent roads
    - max_road_failure_prob:  Worst-case road failure probability
    - is_isolated:           Whether district is disconnected from graph
    - nearest_warehouse_hops: Minimum hops to nearest warehouse node

    Args:
        G: Road network graph.
        district_list: List of district names to extract features for.

    Returns:
        pd.DataFrame with network features per district.
    """
    # Betweenness centrality (expensive for large graphs — use approximation)
    n = G.number_of_nodes()
    if n > 200:
        betweenness = nx.betweenness_centrality(G, k=min(100, n), weight='weight')
    else:
        betweenness = nx.betweenness_centrality(G, weight='weight')

    # Degree centrality
    degree = dict(G.degree())

    rows = []
    for district in district_list:
        if district not in G:
            rows.append({'district': district, 'network_degree': 0,
                         'betweenness_centrality': 0, 'avg_road_failure_prob': 1.0,
                         'max_road_failure_prob': 1.0, 'is_isolated': True})
            continue

        neighbors = list(G.neighbors(district))
        edge_fail_probs = [
            G[district][nb].get('failure_probability', 0) for nb in neighbors
        ]

        rows.append({
            'district':               district,
            'network_degree':         degree.get(district, 0),
            'betweenness_centrality': round(betweenness.get(district, 0), 6),
            'avg_road_failure_prob':  round(np.mean(edge_fail_probs) if edge_fail_probs else 1.0, 4),
            'max_road_failure_prob':  round(max(edge_fail_probs) if edge_fail_probs else 1.0, 4),
            'is_isolated':            degree.get(district, 0) == 0
        })

    return pd.DataFrame(rows)


# --- Extract features ---
network_features = extract_network_features(road_graph, list(infra_gdf['district']))
display(network_features.head(10))

# --- Visualize road network ---
plt.figure(figsize=(12, 10))
pos = {
    node: (data['lon'], data['lat'])
    for node, data in road_graph.nodes(data=True)
    if 'lat' in data and 'lon' in data and data['lat'] != 0
}

if pos:
    node_colors = [
        'red' if road_graph.nodes[n].get('num_warehouses', 0) > 0 else 'steelblue'
        for n in pos.keys()
    ]
    edge_colors = [
        plt.cm.RdYlGn(1 - road_graph[u][v].get('failure_probability', 0))
        for u, v in road_graph.edges() if u in pos and v in pos
    ]

    nx.draw_networkx_nodes(road_graph, pos, nodelist=list(pos.keys()),
                           node_color=node_colors, node_size=80, alpha=0.8)
    nx.draw_networkx_edges(road_graph, pos, edge_color=edge_colors,
                           alpha=0.5, width=1.2)
    nx.draw_networkx_labels(road_graph, pos, labels={n: n for n in pos},
                            font_size=6, font_color='black')

plt.title('District Road Network\n(Red nodes = warehouses | Edge color: green=safe, red=high failure risk)',
          fontsize=11)
plt.axis('off')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'road_network_graph.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Section 9 — Feature Engineering

### Building the district-level feature matrix

Every feature below is a **decision-relevant signal** for the ReliefNet AI engine:

| Feature | Formula / Source | Why it matters |
|---|---|---|
| `flood_risk_score` | Elevation + river proximity + historical floods | Core routing constraint |
| `population_density` | Population / area | Determines aid quantity |
| `vulnerability_score` | Composite (see Section 10) | Priority ranking |
| `road_failure_probability` | From graph analysis | Truck vs UAV decision |
| `warehouse_distance_km` | From spatial join | Delivery time estimate |
| `shelter_capacity_per_1000` | Capacity / population | Adequacy indicator |
| `hospital_density` | Hospitals / area | Medical urgency weight |
| `telecom_density` | Towers / area | Communication reliability |
| `transport_accessibility` | Airport + railway score | Multi-modal logistics |
| `disaster_frequency` | From EM-DAT | Historical risk |
| `historical_damage_score` | Normalized total damage | Economic impact |
| `health_vulnerability` | NFHS-5 composite | Humanitarian priority |

In [ ]:
# ============================================================
# CELL 9.1 — District Feature Engineering
# ============================================================

def engineer_district_features(
    infra_gdf: gpd.GeoDataFrame,
    emdat_df: pd.DataFrame,
    network_features: pd.DataFrame
) -> pd.DataFrame:
    """
    Build the complete district-level feature matrix by combining:
    1. Infrastructure data (infra_gdf)
    2. Historical disaster statistics (emdat_df)
    3. Road network features (network_features)

    Returns:
        pd.DataFrame — one row per district, all engineered features.
    """
    features = infra_gdf.drop(columns=['geometry']).copy()

    # ==========================================================
    # FEATURE 1: Population Density
    # Formula: population / area_sq_km
    # WHY: Dense districts have higher casualty risk per disaster event.
    #      Also determines aid volume requirements.
    # ==========================================================
    features['population_density'] = (
        features['population'] / features['area_sq_km'].replace(0, np.nan)
    ).fillna(0)

    # ==========================================================
    # FEATURE 2: Hospital Density
    # Formula: num_hospitals / area_sq_km × 1000
    # WHY: Low density = underserved areas needing mobile medical units.
    # ==========================================================
    features['hospital_density_per_1000sqkm'] = (
        features['num_hospitals'] / features['area_sq_km'].replace(0, np.nan) * 1000
    ).fillna(0)

    # ==========================================================
    # FEATURE 3: Shelter Adequacy
    # Formula: shelter_capacity_total / (population / 1000)
    # WHY: Insufficient shelter capacity means districts need more
    #      tarpaulins and temporary structures in relief operations.
    # ==========================================================
    features['shelter_capacity_per_1000'] = (
        features['shelter_capacity_total'] /
        (features['population'] / 1000).replace(0, np.nan)
    ).fillna(0)

    # ==========================================================
    # FEATURE 4: Telecom Density
    # Formula: num_telecom_towers / area_sq_km × 100
    # WHY: Low telecom density = communication blackout risk in disasters,
    #      making coordination harder and UAV telemetry unreliable.
    # ==========================================================
    features['telecom_density_per_100sqkm'] = (
        features['num_telecom_towers'] / features['area_sq_km'].replace(0, np.nan) * 100
    ).fillna(0)

    # ==========================================================
    # FEATURE 5: Transport Accessibility Score
    # Formula: weighted combination of road, rail, air access
    # WHY: Multi-modal access determines which logistics vehicles
    #      can reach the district (trucks, trains, cargo flights).
    # ==========================================================
    features['transport_accessibility'] = (
        0.5 * (features['road_length_km'] / features['road_length_km'].max()).fillna(0) +
        0.3 * features.get('has_railway', pd.Series(0, index=features.index)).astype(float) +
        0.2 * features.get('has_airport',  pd.Series(0, index=features.index)).astype(float)
    )

    # ==========================================================
    # FEATURE 6: Flood Risk Score
    # Formula: inverse elevation weight + river proximity weight
    # WHY: Flood-prone districts need pre-positioned boats, pumps,
    #      life jackets, and elevated depot locations.
    # ==========================================================
    elev_norm = (features['avg_elevation_m'] / features['avg_elevation_m'].max()).clip(0, 1)
    riv_norm  = (1 - features['river_proximity_km'] / 200).clip(0, 1)  # closer = higher risk
    coastal   = features.get('coastal_district', pd.Series(0, index=features.index)).astype(float)

    features['flood_risk_score'] = (
        0.4 * (1 - elev_norm) +
        0.4 * riv_norm +
        0.2 * coastal
    ).clip(0, 1)

    # ==========================================================
    # FEATURE 7: Disaster Frequency & Historical Damage
    # Source: EM-DAT aggregated by district
    # WHY: Historically frequent disasters justify pre-stocking
    #      and permanent relief infrastructure investment.
    # ==========================================================
    if 'district_canonical' in emdat_df.columns and 'district' in emdat_df.columns:
        district_col = 'district_canonical' if 'district_canonical' in emdat_df.columns else 'district'
    else:
        district_col = 'district'

    emdat_agg = emdat_df.groupby(district_col).agg(
        disaster_frequency     = (district_col, 'count'),
        total_deaths_historical = ('total_deaths', 'sum'),
        total_affected_historical = ('total_affected', 'sum'),
        total_damage_usd_historical = ('total_damage_usd', 'sum'),
        avg_recency_weight     = ('recency_weight', 'mean') if 'recency_weight' in emdat_df.columns else ('dis_no', 'count')
    ).reset_index().rename(columns={district_col: 'district'})

    # Normalize damage to 0-1
    max_damage = emdat_agg['total_damage_usd_historical'].max()
    emdat_agg['historical_damage_score'] = (
        emdat_agg['total_damage_usd_historical'] / max_damage
    ).fillna(0)

    features = features.merge(emdat_agg, on='district', how='left')
    features['disaster_frequency']      = features['disaster_frequency'].fillna(0)
    features['historical_damage_score'] = features['historical_damage_score'].fillna(0)

    # ==========================================================
    # FEATURE 8: Health Vulnerability Index
    # Source: NFHS-5 indicators
    # WHY: Malnourished / anemic populations have lower disaster
    #      resilience, requiring nutritional supplements in relief.
    # ==========================================================
    nfhs_cols = [c for c in features.columns if c.startswith('nfhs_')]
    if nfhs_cols:
        nfhs_norm = features[nfhs_cols].apply(
            lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
        )
        features['health_vulnerability'] = nfhs_norm.mean(axis=1)
    else:
        features['health_vulnerability'] = 0.0

    # ==========================================================
    # FEATURE 9: Warehouse Supply Score
    # Formula: total supply tonnage / population
    # WHY: Determines how long existing stock can sustain relief
    #      without resupply — critical for logistics planning.
    # ==========================================================
    supply_cols = [c for c in features.columns if c.startswith('warehouse_')]
    if supply_cols:
        total_supply = features[supply_cols].sum(axis=1)
        features['warehouse_supply_per_capita'] = (
            total_supply / features['population'].replace(0, np.nan)
        ).fillna(0)

    # ==========================================================
    # Merge road network features
    # ==========================================================
    features = features.merge(network_features, on='district', how='left')
    features['avg_road_failure_prob'] = features['avg_road_failure_prob'].fillna(1.0)
    features['is_isolated']           = features['is_isolated'].fillna(True)

    logger.info(f'Feature matrix created: {features.shape}')
    return features


# --- Build feature matrix ---
feature_matrix = engineer_district_features(infra_gdf, emdat_df, network_features)

print(f'✅ Feature matrix: {feature_matrix.shape}')
print(f'   Columns: {list(feature_matrix.columns)}')
display(feature_matrix.head(5))

In [ ]:
# ============================================================
# CELL 9.2 — Feature Distribution Visualization
# ============================================================

key_features = [
    'flood_risk_score', 'population_density', 'hospital_density_per_1000sqkm',
    'shelter_capacity_per_1000', 'telecom_density_per_100sqkm',
    'transport_accessibility', 'disaster_frequency', 'historical_damage_score',
    'health_vulnerability', 'avg_road_failure_prob'
]

existing_key = [c for c in key_features if c in feature_matrix.columns]
plot_distributions(feature_matrix[existing_key], 'Engineered Feature Distributions')
plot_correlation_heatmap(feature_matrix[existing_key], 'Feature Correlation Matrix')

---
# Section 10 — Vulnerability Index Creation

### What is the Vulnerability Index?

The **District Vulnerability Index (DVI)** is a single composite score (0-1) that combines:
- Disaster exposure (flood risk, disaster frequency)
- Physical fragility (road failure, isolation)
- Social fragility (health indicators, shelter inadequacy)
- Adaptive capacity (warehouses, hospitals, telecom)

A district with DVI = 0.85 has **very high vulnerability** → highest priority in resource allocation.

### Why PCA?
Many vulnerability sub-indicators are correlated (e.g., flood risk and road failure both relate to water levels). PCA removes redundancy while preserving the maximum explained variance — preventing double-counting in the composite score.

In [ ]:
# ============================================================
# CELL 10.1 — Composite Vulnerability Index
# ============================================================

def create_vulnerability_index(
    df: pd.DataFrame,
    use_pca: bool = True,
    n_pca_components: int = 3
) -> pd.DataFrame:
    """
    Compute a District Vulnerability Index (DVI) using:
    1. Weighted sum of normalized risk sub-indicators
    2. PCA-based composite (optional, reduces collinearity)

    DVI Components and Weights:
    ┌─────────────────────────────┬────────┐
    │ Component                   │ Weight │
    ├─────────────────────────────┼────────┤
    │ Flood risk score            │  0.20  │
    │ Disaster frequency (norm)   │  0.15  │
    │ Historical damage (norm)    │  0.10  │
    │ Road failure probability    │  0.15  │
    │ Health vulnerability        │  0.15  │
    │ Shelter inadequacy          │  0.10  │
    │ Hospital density (inverse)  │  0.10  │
    │ Telecom density (inverse)   │  0.05  │
    └─────────────────────────────┴────────┘

    Args:
        df: Feature matrix DataFrame.
        use_pca: If True, compute PCA-based DVI in addition to weighted.
        n_pca_components: Number of PCA components to retain.

    Returns:
        DataFrame with 'vulnerability_score' and 'vulnerability_tier' added.
    """
    df = df.copy()
    scaler = MinMaxScaler()

    # Sub-indicator configuration: (column, weight, invert?)
    # invert=True means HIGHER value = LESS vulnerable (e.g., more hospitals = safer)
    VULNERABILITY_COMPONENTS = [
        ('flood_risk_score',                  0.20, False),
        ('disaster_frequency',                0.15, False),
        ('historical_damage_score',           0.10, False),
        ('avg_road_failure_prob',             0.15, False),
        ('health_vulnerability',              0.15, False),
        ('shelter_capacity_per_1000',         0.10, True),  # more shelter = less vulnerable
        ('hospital_density_per_1000sqkm',     0.10, True),  # more hospitals = less vulnerable
        ('telecom_density_per_100sqkm',       0.05, True),  # more towers = less vulnerable
    ]

    vi_score = np.zeros(len(df))
    used_cols = []

    for col, weight, invert in VULNERABILITY_COMPONENTS:
        if col not in df.columns:
            logger.warning(f'Vulnerability component {col} not found — skipping')
            continue

        values = df[col].fillna(0).values.reshape(-1, 1)
        norm   = scaler.fit_transform(values).flatten()

        if invert:
            norm = 1 - norm  # flip: high availability → low vulnerability

        vi_score += weight * norm
        used_cols.append(col)

    df['vulnerability_score'] = vi_score.clip(0, 1).round(4)

    # --- PCA-based composite ---
    if use_pca and len(used_cols) >= n_pca_components:
        pca_data = df[used_cols].fillna(0)
        pca_scaled = StandardScaler().fit_transform(pca_data)

        pca = PCA(n_components=n_pca_components, random_state=42)
        pca_scores = pca.fit_transform(pca_scaled)

        # Use first component as PCA vulnerability signal
        pca_vi = pca_scores[:, 0]
        # Normalize to 0-1
        pca_vi = (pca_vi - pca_vi.min()) / (pca_vi.max() - pca_vi.min() + 1e-9)
        df['vulnerability_score_pca'] = pca_vi.round(4)

        explained = pca.explained_variance_ratio_
        logger.info(f'PCA explained variance: {[round(v,3) for v in explained]}')
        print(f'  PCA explained variance ratio: {[round(v,3) for v in explained]}')

    # --- Vulnerability Tier Classification ---
    def classify_tier(score: float) -> str:
        if score >= 0.75: return 'CRITICAL'
        if score >= 0.55: return 'HIGH'
        if score >= 0.35: return 'MEDIUM'
        return 'LOW'

    df['vulnerability_tier'] = df['vulnerability_score'].apply(classify_tier)

    tier_counts = df['vulnerability_tier'].value_counts()
    print('\n📊 Vulnerability Tier Distribution:')
    print(tier_counts.to_string())

    logger.info(f'Vulnerability index computed for {len(df)} districts')
    return df


# --- Compute DVI ---
feature_matrix = create_vulnerability_index(feature_matrix)

# --- Visualize ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(feature_matrix['vulnerability_score'], bins=15,
             color='tomato', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribution of Vulnerability Scores', fontweight='bold')
axes[0].set_xlabel('Vulnerability Score (0=Low, 1=Critical)')
axes[0].axvline(0.55, color='orange', linestyle='--', label='HIGH threshold')
axes[0].axvline(0.75, color='red',    linestyle='--', label='CRITICAL threshold')
axes[0].legend()

# Tier pie
tier_counts = feature_matrix['vulnerability_tier'].value_counts()
colors_map = {'CRITICAL': 'red', 'HIGH': 'orange', 'MEDIUM': 'gold', 'LOW': 'steelblue'}
axes[1].pie(
    tier_counts.values,
    labels=tier_counts.index,
    autopct='%1.0f%%',
    colors=[colors_map.get(t, 'gray') for t in tier_counts.index],
    startangle=140
)
axes[1].set_title('Districts by Vulnerability Tier', fontweight='bold')

plt.tight_layout()
plt.savefig(REPORT_DIR / 'vulnerability_index.png', dpi=120)
plt.show()

---
# Section 11 — UAV + Logistics Feature Preparation

### UAV-specific features for ReliefNet

UAVs and trucks have different operational constraints:

| Constraint | Truck | UAV |
|---|---|---|
| Road dependency | REQUIRED | None |
| Range | 500-1000 km/day | 15-100 km per flight |
| Payload | 5-20 tons | 5-50 kg |
| Terrain | Road quality matters | Elevation matters |
| Monsoon impact | Road flooding | Wind speed limits |

Features in this section help the optimizer decide:
**"Should we send a truck, a UAV, or both?"**

In [ ]:
# ============================================================
# CELL 11.1 — UAV & Logistics Feature Engineering
# ============================================================

def engineer_uav_logistics_features(
    df: pd.DataFrame,
    uav_max_range_km: float = 60.0,
    truck_speed_kmh: float  = 40.0,
) -> pd.DataFrame:
    """
    Compute UAV and truck logistics suitability features per district.

    Features:
    - uav_reachable_within_range: Is district within UAV range from nearest warehouse?
    - uav_terrain_penalty:        Elevation-based UAV performance penalty (>3000m = difficult)
    - truck_estimated_hours:       Estimated truck delivery time (warehouse distance / speed)
    - truck_accessibility_score:   Composite truck suitability (roads + no isolation)
    - disconnected_region_score:   Probability district is cut off after disaster
    - preferred_mode:             Recommendation: 'truck', 'uav', or 'both'

    Args:
        df: Feature matrix.
        uav_max_range_km: Maximum UAV operational radius in km.
        truck_speed_kmh:  Average truck speed on disaster-affected roads.

    Returns:
        DataFrame with logistics features added.
    """
    df = df.copy()

    # --- UAV Reachability ---
    if 'nearest_warehouse_km' in df.columns:
        df['uav_reachable_within_range'] = (
            df['nearest_warehouse_km'] <= uav_max_range_km
        ).astype(int)
    else:
        df['uav_reachable_within_range'] = 0

    # --- UAV Terrain Penalty ---
    # Above 2000m: significant performance penalty (thinner air, wind)
    # Above 4000m: UAV operations not feasible with standard drones
    if 'avg_elevation_m' in df.columns:
        df['uav_terrain_penalty'] = (
            df['avg_elevation_m'].clip(0, 5000) / 5000
        ).clip(0, 1)
    else:
        df['uav_terrain_penalty'] = 0.0

    # --- Truck Delivery Time Estimate ---
    if 'nearest_warehouse_km' in df.columns:
        df['truck_estimated_hours'] = (
            df['nearest_warehouse_km'] / truck_speed_kmh
        ).round(2)

    # --- Truck Accessibility Score ---
    # Combines: road length, connectivity, failure probability, isolation status
    road_norm = (df.get('road_length_km', pd.Series(500, index=df.index)) /
                 df.get('road_length_km', pd.Series(500, index=df.index)).max()).clip(0, 1)

    fail_prob = df.get('avg_road_failure_prob', pd.Series(0.5, index=df.index)).fillna(0.5)
    isolated  = df.get('is_isolated', pd.Series(False, index=df.index)).astype(float)

    df['truck_accessibility_score'] = (
        0.4 * road_norm +
        0.4 * (1 - fail_prob) +
        0.2 * (1 - isolated)
    ).clip(0, 1).round(4)

    # --- Disconnected Region Score ---
    # High score = likely to be isolated after disaster
    df['disconnected_region_score'] = (
        0.5 * fail_prob +
        0.3 * df.get('flood_risk_score', pd.Series(0, index=df.index)).fillna(0) +
        0.2 * isolated
    ).clip(0, 1).round(4)

    # --- Preferred Delivery Mode ---
    def determine_mode(row: pd.Series) -> str:
        truck_ok = row.get('truck_accessibility_score', 0) > 0.5
        uav_ok   = (row.get('uav_reachable_within_range', 0) == 1 and
                    row.get('uav_terrain_penalty', 1) < 0.5)
        if truck_ok and uav_ok:
            return 'both'
        elif uav_ok:
            return 'uav'
        elif truck_ok:
            return 'truck'
        else:
            return 'emergency_airlift'

    df['preferred_delivery_mode'] = df.apply(determine_mode, axis=1)

    mode_counts = df['preferred_delivery_mode'].value_counts()
    print('\n🚛✈️  Preferred Delivery Mode Distribution:')
    print(mode_counts.to_string())

    logger.info('UAV & logistics features engineered.')
    return df


feature_matrix = engineer_uav_logistics_features(feature_matrix)

# --- Visualize delivery mode ---
mode_counts = feature_matrix['preferred_delivery_mode'].value_counts()
fig = px.pie(
    values=mode_counts.values, names=mode_counts.index,
    title='Recommended Delivery Mode per District',
    color_discrete_map={
        'both': 'steelblue', 'truck': 'green',
        'uav': 'purple', 'emergency_airlift': 'red'
    }
)
fig.show()

---
# Section 12 — MongoDB Integration

### Why MongoDB for ReliefNet?

MongoDB is ideal for this project because:
1. **Schema flexibility** — disaster events vary in fields by type (flood vs earthquake)
2. **Geospatial indexing** — native 2dsphere indexes for location queries
3. **Aggregation pipelines** — efficient district-level roll-ups in real time
4. **Time-series collections** — built-in support for disaster timeline data
5. **Horizontal scalability** — handles high-volume real-time disaster telemetry

Collections:
- `districts` — district features + vulnerability index
- `warehouses` — warehouse locations + inventory
- `disasters` — EM-DAT events + enriched features
- `road_network` — edge list with failure probabilities
- `shelters` — shelter locations + capacities
- `hospitals` — hospital locations + specialties
- `simulation_features` — unified ML-ready feature matrix

In [ ]:
# ============================================================
# CELL 12.1 — MongoDB Schema & Insert Pipeline
# ============================================================

# ─── MongoDB Configuration ────────────────────────────────
MONGO_URI = os.getenv('MONGO_URI', 'mongodb://localhost:27017/')
MONGO_DB  = os.getenv('MONGO_DB',  'reliefnet')
# ──────────────────────────────────────────────────────────


class ReliefNetMongoPipeline:
    """
    Production-grade MongoDB integration pipeline for ReliefNet.

    Handles:
    - Connection management
    - Collection creation with indexes
    - Batch upsert with duplicate prevention
    - Schema validation
    - Geospatial indexing
    """

    COLLECTION_SCHEMAS = {
        'districts': {
            '$jsonSchema': {
                'bsonType': 'object',
                'required': ['district', 'state', 'vulnerability_score'],
                'properties': {
                    'district':            {'bsonType': 'string'},
                    'state':               {'bsonType': 'string'},
                    'vulnerability_score': {'bsonType': 'double', 'minimum': 0, 'maximum': 1},
                    'location': {
                        'bsonType': 'object',
                        'properties': {
                            'type':        {'bsonType': 'string'},
                            'coordinates': {'bsonType': 'array'}
                        }
                    }
                }
            }
        },
        'disasters': {
            '$jsonSchema': {
                'bsonType': 'object',
                'required': ['dis_no', 'year', 'disaster_type'],
                'properties': {
                    'dis_no':       {'bsonType': 'string'},
                    'year':         {'bsonType': ['int', 'double']},
                    'disaster_type':{'bsonType': 'string'},
                }
            }
        }
    }

    def __init__(self, uri: str = MONGO_URI, db_name: str = MONGO_DB):
        self.uri     = uri
        self.db_name = db_name
        self.client  = None
        self.db      = None

    def connect(self) -> bool:
        """Establish MongoDB connection with timeout."""
        try:
            self.client = MongoClient(self.uri, serverSelectionTimeoutMS=5000)
            self.client.server_info()  # Triggers connection test
            self.db = self.client[self.db_name]
            logger.info(f'MongoDB connected: {self.db_name}')
            return True
        except Exception as e:
            logger.warning(f'MongoDB connection failed: {e}')
            return False

    def setup_collections(self) -> None:
        """Create collections with schema validation and indexes."""
        collection_configs = [
            {
                'name': 'districts',
                'indexes': [
                    [('district', ASCENDING), ('state', ASCENDING)],
                    [('vulnerability_score', ASCENDING)],
                    [('location', GEOSPHERE)]
                ],
                'unique_field': 'district'
            },
            {
                'name': 'disasters',
                'indexes': [
                    [('dis_no', ASCENDING)],
                    [('year', ASCENDING)],
                    [('disaster_type', ASCENDING)],
                    [('location', GEOSPHERE)]
                ],
                'unique_field': 'dis_no'
            },
            {
                'name': 'warehouses',
                'indexes': [
                    [('warehouse_id', ASCENDING)],
                    [('location', GEOSPHERE)]
                ],
                'unique_field': 'warehouse_id'
            },
            {
                'name': 'road_network',
                'indexes': [[('source', ASCENDING), ('target', ASCENDING)]],
                'unique_field': None
            },
            {
                'name': 'shelters',
                'indexes': [
                    [('shelter_id', ASCENDING)],
                    [('location', GEOSPHERE)]
                ],
                'unique_field': 'shelter_id'
            },
            {
                'name': 'hospitals',
                'indexes': [
                    [('hospital_id', ASCENDING)],
                    [('location', GEOSPHERE)]
                ],
                'unique_field': 'hospital_id'
            },
            {
                'name': 'simulation_features',
                'indexes': [
                    [('district', ASCENDING)],
                    [('vulnerability_tier', ASCENDING)]
                ],
                'unique_field': 'district'
            }
        ]

        existing_cols = self.db.list_collection_names()
        for config in collection_configs:
            name = config['name']
            if name not in existing_cols:
                self.db.create_collection(name)
                logger.info(f'Collection created: {name}')

            col = self.db[name]
            for index_spec in config['indexes']:
                try:
                    col.create_index(index_spec)
                except Exception as e:
                    logger.debug(f'Index creation note for {name}: {e}')

            if config['unique_field']:
                try:
                    col.create_index(
                        [(config['unique_field'], ASCENDING)],
                        unique=True, name=f'unique_{config["unique_field"]}'
                    )
                except Exception:
                    pass

        logger.info('All collections and indexes configured.')

    def df_to_mongo_docs(
        self,
        df: pd.DataFrame,
        lat_col: Optional[str] = None,
        lon_col: Optional[str] = None
    ) -> List[Dict]:
        """
        Convert DataFrame to list of MongoDB documents.
        Adds GeoJSON 'location' field if lat/lon columns provided.
        Converts NaN to None for MongoDB compatibility.

        Args:
            df: Source DataFrame.
            lat_col: Latitude column name.
            lon_col: Longitude column name.

        Returns:
            List of document dicts.
        """
        df = df.copy()
        # Convert numpy types to Python native for BSON compatibility
        for col in df.select_dtypes(include=[np.integer]).columns:
            df[col] = df[col].astype(int)
        for col in df.select_dtypes(include=[np.floating]).columns:
            df[col] = df[col].astype(float)

        records = df.where(df.notna(), None).to_dict('records')

        if lat_col and lon_col:
            for rec in records:
                lat = rec.get(lat_col)
                lon = rec.get(lon_col)
                if lat is not None and lon is not None:
                    rec['location'] = {
                        'type': 'Point',
                        'coordinates': [lon, lat]  # GeoJSON: [lon, lat]
                    }
        return records

    def batch_upsert(
        self,
        collection_name: str,
        documents: List[Dict],
        unique_key: str,
        batch_size: int = 500
    ) -> Dict[str, int]:
        """
        Upsert documents in batches to prevent duplicates.
        Uses MongoDB replace_one with upsert=True.

        Args:
            collection_name: Target collection.
            documents: List of document dicts.
            unique_key: Field used as unique identifier.
            batch_size: Documents per batch.

        Returns:
            Dict with 'inserted', 'updated', 'errors' counts.
        """
        from pymongo import ReplaceOne

        col = self.db[collection_name]
        stats = {'inserted': 0, 'updated': 0, 'errors': 0}

        for batch_start in range(0, len(documents), batch_size):
            batch = documents[batch_start:batch_start + batch_size]
            operations = [
                ReplaceOne(
                    {unique_key: doc.get(unique_key)},
                    doc,
                    upsert=True
                )
                for doc in batch if doc.get(unique_key) is not None
            ]

            if not operations:
                continue

            try:
                result = col.bulk_write(operations, ordered=False)
                stats['inserted'] += result.upserted_count
                stats['updated']  += result.modified_count
            except BulkWriteError as bwe:
                stats['errors'] += len(bwe.details.get('writeErrors', []))
                logger.warning(f'Bulk write partial error: {len(bwe.details.get("writeErrors", []))} errors')

        logger.info(f'[{collection_name}] Upsert: {stats}')
        return stats

    def close(self) -> None:
        """Close MongoDB connection."""
        if self.client:
            self.client.close()
            logger.info('MongoDB connection closed.')


print('✅ MongoDB pipeline class defined.')

In [ ]:
# ============================================================
# CELL 12.2 — Run MongoDB Pipeline
# ============================================================

mongo = ReliefNetMongoPipeline()
mongo_available = mongo.connect()

if mongo_available:
    # --- Setup collections + indexes ---
    mongo.setup_collections()

    # --- Insert: Districts ---
    district_docs = mongo.df_to_mongo_docs(
        feature_matrix.drop(columns=[c for c in feature_matrix.columns
                                      if 'historical' in c and 'score' not in c], errors='ignore'),
        lat_col='latitude', lon_col='longitude'
    )
    stats = mongo.batch_upsert('districts', district_docs, unique_key='district')
    print(f'districts: {stats}')

    # --- Insert: Disasters ---
    # Convert datetime to string for MongoDB
    emdat_mongo = emdat_df.copy()
    for col in emdat_mongo.select_dtypes(include=['datetime64[ns]']).columns:
        emdat_mongo[col] = emdat_mongo[col].dt.strftime('%Y-%m-%d')

    disaster_docs = mongo.df_to_mongo_docs(
        emdat_mongo, lat_col='latitude', lon_col='longitude'
    )
    stats = mongo.batch_upsert('disasters', disaster_docs, unique_key='dis_no')
    print(f'disasters: {stats}')

    # --- Insert: Road Network Edges ---
    edge_records = [
        {
            'source':              u,
            'target':              v,
            'distance_km':         data.get('distance_km', 0),
            'flood_risk':          data.get('flood_risk', 0),
            'landslide_risk':      data.get('landslide_risk', 0),
            'failure_probability': data.get('failure_probability', 0)
        }
        for u, v, data in road_graph.edges(data=True)
    ]
    if edge_records:
        mongo.db['road_network'].drop()
        mongo.db['road_network'].insert_many(edge_records)
        print(f'road_network: {len(edge_records)} edges inserted')

    # --- Insert: Simulation Features ---
    sim_docs = mongo.df_to_mongo_docs(
        feature_matrix, lat_col='latitude', lon_col='longitude'
    )
    stats = mongo.batch_upsert('simulation_features', sim_docs, unique_key='district')
    print(f'simulation_features: {stats}')

    mongo.close()
    print('\n✅ All data inserted into MongoDB.')

else:
    print('⚠️  MongoDB not available. Skipping DB insertion.')
    print('   Set MONGO_URI env var to connect. Continuing to export section...')

---
# Section 13 — Data Validation Layer

### Why validation is mandatory before any ML pipeline

A validation layer catches issues that preprocessing might have missed:
- **Coordinate drift** — centroid lands in the ocean (wrong state assigned)
- **Score overflow** — vulnerability score > 1.0 due to calculation error
- **Duplicate district IDs** — two rows for same district with different data
- **Outlier contamination** — imputed values that are physically impossible
- **Schema mismatch** — column expected by ML model has wrong data type

In [ ]:
# ============================================================
# CELL 13.1 — Validation Framework
# ============================================================

def validate_feature_matrix(
    df: pd.DataFrame,
    dataset_name: str = 'Feature Matrix'
) -> Dict[str, Any]:
    """
    Run a comprehensive validation suite on the final feature matrix.

    Checks:
    1. Schema completeness (required columns present)
    2. Null percentage per column (must be < 5% for core features)
    3. Coordinate validity (within India bounding box)
    4. Score bounds (0-1 for normalized features)
    5. Duplicate district detection
    6. Outlier detection (IQR method)
    7. Type conformance

    Args:
        df: Final feature matrix.
        dataset_name: Label for report.

    Returns:
        Dict with validation results and issue counts.
    """
    report = {
        'dataset': dataset_name,
        'total_rows': len(df),
        'total_columns': len(df.columns),
        'issues': []
    }

    # --- 1. Required columns ---
    REQUIRED_COLS = [
        'district', 'state', 'population', 'vulnerability_score',
        'flood_risk_score', 'transport_accessibility'
    ]
    missing_required = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing_required:
        report['issues'].append({
            'type': 'MISSING_REQUIRED_COLS',
            'severity': 'CRITICAL',
            'detail': missing_required
        })

    # --- 2. Null check on core features ---
    CORE_FEATURES = [c for c in REQUIRED_COLS if c in df.columns]
    for col in CORE_FEATURES:
        null_pct = df[col].isna().mean() * 100
        if null_pct > 5:
            report['issues'].append({
                'type': 'HIGH_NULL_CORE_FEATURE',
                'severity': 'HIGH',
                'detail': f'{col}: {null_pct:.1f}% null'
            })

    # --- 3. Coordinate validation ---
    if 'latitude' in df.columns and 'longitude' in df.columns:
        bad_lat = df[(df['latitude'] < 6) | (df['latitude'] > 38)]
        bad_lon = df[(df['longitude'] < 67) | (df['longitude'] > 99)]
        if not bad_lat.empty:
            report['issues'].append({
                'type': 'INVALID_COORDINATES',
                'severity': 'MEDIUM',
                'detail': f'{len(bad_lat)} rows with latitude outside India (6–38°N)'
            })
        if not bad_lon.empty:
            report['issues'].append({
                'type': 'INVALID_COORDINATES',
                'severity': 'MEDIUM',
                'detail': f'{len(bad_lon)} rows with longitude outside India (67–99°E)'
            })

    # --- 4. Score bounds check ---
    SCORE_COLS = [
        'vulnerability_score', 'flood_risk_score', 'transport_accessibility',
        'avg_road_failure_prob', 'truck_accessibility_score',
        'disconnected_region_score'
    ]
    for col in SCORE_COLS:
        if col not in df.columns:
            continue
        out_of_bounds = df[(df[col] < 0) | (df[col] > 1)]
        if not out_of_bounds.empty:
            report['issues'].append({
                'type': 'SCORE_OUT_OF_BOUNDS',
                'severity': 'HIGH',
                'detail': f'{col}: {len(out_of_bounds)} values outside [0,1]'
            })

    # --- 5. Duplicate districts ---
    if 'district' in df.columns:
        dups = df[df['district'].duplicated()]
        if not dups.empty:
            report['issues'].append({
                'type': 'DUPLICATE_DISTRICTS',
                'severity': 'CRITICAL',
                'detail': f'{len(dups)} duplicate district entries: {list(dups["district"].unique())}'
            })

    # --- 6. Outlier detection (IQR method) ---
    OUTLIER_CHECK_COLS = ['population', 'population_density', 'disaster_frequency']
    outlier_summary = {}
    for col in OUTLIER_CHECK_COLS:
        if col not in df.columns:
            continue
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers = df[(df[col] < Q1 - 3 * IQR) | (df[col] > Q3 + 3 * IQR)]
        if not outliers.empty:
            outlier_summary[col] = len(outliers)

    if outlier_summary:
        report['issues'].append({
            'type': 'EXTREME_OUTLIERS',
            'severity': 'LOW',
            'detail': str(outlier_summary)
        })

    # --- Summary ---
    critical = sum(1 for i in report['issues'] if i['severity'] == 'CRITICAL')
    high     = sum(1 for i in report['issues'] if i['severity'] == 'HIGH')
    medium   = sum(1 for i in report['issues'] if i['severity'] == 'MEDIUM')
    low      = sum(1 for i in report['issues'] if i['severity'] == 'LOW')

    report['summary'] = {
        'CRITICAL': critical, 'HIGH': high, 'MEDIUM': medium, 'LOW': low,
        'TOTAL_ISSUES': len(report['issues'])
    }

    # Print report
    print(f'\n{'='*55}')
    print(f'  🔍 Validation Report: {dataset_name}')
    print(f'{'='*55}')
    print(f'  Rows: {report["total_rows"]} | Columns: {report["total_columns"]}')
    print(f'  Issues: CRITICAL={critical} | HIGH={high} | MEDIUM={medium} | LOW={low}')
    print()
    for issue in report['issues']:
        icon = {'CRITICAL': '🚨', 'HIGH': '⚠️', 'MEDIUM': '⚡', 'LOW': 'ℹ️'}.get(issue['severity'], '?')
        print(f'  {icon} [{issue["severity"]}] {issue["type"]}: {issue["detail"]}')

    if not report['issues']:
        print('  ✅ All validation checks passed!')

    # Save report
    import json as _json
    with open(REPORT_DIR / 'validation_report.json', 'w') as f:
        _json.dump(report, f, indent=2, default=str)

    return report


validation_report = validate_feature_matrix(feature_matrix, 'ReliefNet Feature Matrix')

---
# Section 14 — Final Unified Dataset Export

### Output formats

| Format | Use Case |
|---|---|
| **CSV** | Human inspection, Excel viewing, lightweight sharing |
| **Parquet** | Columnar format for pandas / Spark ML pipelines (3-10x smaller than CSV) |
| **JSON** | API serving, frontend integration |
| **MongoDB** | Real-time query, geospatial operations, simulation engine |

Parquet is the recommended format for all downstream ML models due to:
- Columnar compression (10-100x faster for column-wise ML operations)
- Schema preservation (data types are stored)
- Partitioning support (by state/year for large-scale processing)

In [ ]:
# ============================================================
# CELL 14.1 — Export Pipeline
# ============================================================

def export_final_datasets(
    feature_matrix: pd.DataFrame,
    emdat_df: pd.DataFrame,
    road_graph: nx.Graph,
    export_dir: Path = EXPORT_DIR
) -> Dict[str, Path]:
    """
    Export all final datasets to disk in multiple formats.

    Args:
        feature_matrix: Unified district feature matrix.
        emdat_df: Cleaned disaster events.
        road_graph: Road network NetworkX graph.
        export_dir: Output directory.

    Returns:
        Dict mapping dataset name to file path.
    """
    export_dir.mkdir(parents=True, exist_ok=True)
    exported = {}

    # --- Feature Matrix ---
    csv_path = export_dir / 'reliefnet_district_features.csv'
    feature_matrix.to_csv(csv_path, index=False)
    exported['district_features_csv'] = csv_path
    logger.info(f'Exported: {csv_path}')

    parquet_path = export_dir / 'reliefnet_district_features.parquet'
    # Convert bool columns for parquet compatibility
    fm_export = feature_matrix.copy()
    for col in fm_export.select_dtypes(include=['bool']).columns:
        fm_export[col] = fm_export[col].astype(int)
    fm_export.to_parquet(parquet_path, index=False, engine='pyarrow')
    exported['district_features_parquet'] = parquet_path
    logger.info(f'Exported: {parquet_path}')

    # --- Disaster Events ---
    emdat_export = emdat_df.copy()
    for col in emdat_export.select_dtypes(include=['datetime64[ns]']).columns:
        emdat_export[col] = emdat_export[col].dt.strftime('%Y-%m-%d')

    emdat_csv = export_dir / 'reliefnet_disasters.csv'
    emdat_export.to_csv(emdat_csv, index=False)
    exported['disasters_csv'] = emdat_csv

    emdat_parquet = export_dir / 'reliefnet_disasters.parquet'
    for col in emdat_export.select_dtypes(include=['bool']).columns:
        emdat_export[col] = emdat_export[col].astype(int)
    emdat_export.to_parquet(emdat_parquet, index=False, engine='pyarrow')
    exported['disasters_parquet'] = emdat_parquet

    # --- Road Network Edge List ---
    edges = [
        {'source': u, 'target': v, **data}
        for u, v, data in road_graph.edges(data=True)
    ]
    if edges:
        edges_df  = pd.DataFrame(edges)
        edges_csv = export_dir / 'reliefnet_road_edges.csv'
        edges_df.to_csv(edges_csv, index=False)
        exported['road_edges_csv'] = edges_csv

    # --- Feature Dictionary ---
    feature_dict = {
        'metadata': {
            'created_at': datetime.now().isoformat(),
            'total_districts': len(feature_matrix),
            'total_features': len(feature_matrix.columns),
            'version': '1.0.0'
        },
        'features': {}
    }

    FEATURE_DESCRIPTIONS = {
        'population_density':           'Population per sq km — determines aid volume requirements',
        'flood_risk_score':             'Composite flood exposure (0-1) based on elevation + river proximity',
        'vulnerability_score':          'District Vulnerability Index (0=Low, 1=Critical)',
        'vulnerability_tier':           'Categorical tier: LOW / MEDIUM / HIGH / CRITICAL',
        'avg_road_failure_prob':        'Mean probability of road failure in disaster scenario (0-1)',
        'transport_accessibility':      'Multi-modal transport score: road + rail + air (0-1)',
        'shelter_capacity_per_1000':    'Evacuation shelter spaces per 1000 population',
        'hospital_density_per_1000sqkm':'Hospitals per 1000 sq km area',
        'telecom_density_per_100sqkm':  'Telecom towers per 100 sq km',
        'disaster_frequency':           'Historical count of disasters affecting district (EM-DAT)',
        'historical_damage_score':      'Normalized economic damage from historical disasters',
        'health_vulnerability':         'Composite NFHS-5 health deprivation index (0-1)',
        'truck_accessibility_score':    'Truck routing suitability score (0-1)',
        'uav_terrain_penalty':          'Terrain difficulty for UAV operations (0=easy, 1=infeasible)',
        'disconnected_region_score':    'Probability of post-disaster road isolation (0-1)',
        'preferred_delivery_mode':      'AI recommendation: truck/uav/both/emergency_airlift',
        'nearest_warehouse_km':         'Euclidean distance to nearest warehouse in km',
        'betweenness_centrality':       'Road network routing criticality score',
    }

    for col in feature_matrix.columns:
        feature_dict['features'][col] = {
            'dtype':       str(feature_matrix[col].dtype),
            'description': FEATURE_DESCRIPTIONS.get(col, 'See pipeline documentation'),
            'null_pct':    round(feature_matrix[col].isna().mean() * 100, 2),
            'min':         float(feature_matrix[col].min()) if pd.api.types.is_numeric_dtype(feature_matrix[col]) else None,
            'max':         float(feature_matrix[col].max()) if pd.api.types.is_numeric_dtype(feature_matrix[col]) else None,
        }

    dict_path = export_dir / 'reliefnet_feature_dictionary.json'
    with open(dict_path, 'w') as f:
        json.dump(feature_dict, f, indent=2, default=str)
    exported['feature_dictionary'] = dict_path

    print('\n✅ Export Complete:')
    for name, path in exported.items():
        size_kb = path.stat().st_size / 1024
        print(f'   {name:40s}  {size_kb:8.1f} KB  →  {path.name}')

    return exported


exported_files = export_final_datasets(feature_matrix, emdat_df, road_graph)

---
# Section 15 — Final Pipeline Summary

## ✅ ReliefNet Data Engineering Pipeline — Complete

This final section summarizes every preprocessing step, validates the pipeline end-to-end, and declares ML-readiness for each downstream module.

In [ ]:
# ============================================================
# CELL 15.1 — Pipeline Summary Report
# ============================================================

def generate_pipeline_summary(
    feature_matrix: pd.DataFrame,
    emdat_df: pd.DataFrame,
    road_graph: nx.Graph,
    validation_report: Dict
) -> None:
    """
    Print and save a complete pipeline summary report.

    Args:
        feature_matrix: Final feature matrix.
        emdat_df: Processed disaster events.
        road_graph: Road network graph.
        validation_report: Validation results dict.
    """
    print('\n' + '='*70)
    print('  🌐 RELIEFNET DATA ENGINEERING PIPELINE — COMPLETION REPORT')
    print('='*70)

    print('\n📦 DATASETS PROCESSED')
    print(f'  EM-DAT Disaster Events    : {len(emdat_df):,} records')
    print(f'  Infrastructure/Health Dist: {len(infra_df):,} districts')
    print(f'  Road Network Nodes        : {road_graph.number_of_nodes()}')
    print(f'  Road Network Edges        : {road_graph.number_of_edges()}')

    print('\n🔧 PREPROCESSING STEPS COMPLETED')
    steps = [
        'CSV/Excel/GeoJSON multi-format loaders with encoding fallback',
        'Automated EDA: profiling, null heatmaps, distributions, correlations',
        'District name standardization via RapidFuzz fuzzy matching',
        'Tiered imputation: state-wise median → KNN → global median fallback',
        'Temporal normalization: ISO-8601, disaster duration, season, recency weight',
        'Geospatial processing: WGS84 normalization, Point geometry creation',
        'Road network graph: node/edge features, failure probability, connectivity',
        'District Vulnerability Index (DVI): weighted composite + PCA variant',
        'UAV & logistics features: terrain penalty, mode recommendation',
        'MongoDB pipeline: upsert, geospatial indexes, batch insert',
        'Validation layer: schema, coordinates, score bounds, duplicates, outliers',
        'Multi-format export: CSV, Parquet, JSON feature dictionary',
    ]
    for i, step in enumerate(steps, 1):
        print(f'  {i:2d}. ✅ {step}')

    print('\n🧠 FEATURES CREATED')
    feat_groups = {
        'Disaster Risk':    ['flood_risk_score', 'disaster_frequency', 'historical_damage_score'],
        'Infrastructure':   ['hospital_density_per_1000sqkm', 'telecom_density_per_100sqkm',
                             'transport_accessibility', 'shelter_capacity_per_1000'],
        'Road Network':     ['avg_road_failure_prob', 'network_degree', 'betweenness_centrality',
                             'is_isolated', 'disconnected_region_score'],
        'Vulnerability':    ['vulnerability_score', 'vulnerability_tier', 'health_vulnerability'],
        'Logistics/UAV':    ['truck_accessibility_score', 'uav_terrain_penalty',
                             'preferred_delivery_mode', 'nearest_warehouse_km'],
        'Demographics':     ['population_density', 'population'],
    }
    for group, cols in feat_groups.items():
        available = [c for c in cols if c in feature_matrix.columns]
        print(f'  {group:18s}: {len(available)} features — {available}')

    print(f'\n  Total Features  : {len(feature_matrix.columns)}')
    print(f'  Total Districts : {len(feature_matrix)}')

    print('\n🤖 DOWNSTREAM ML READINESS')
    modules = [
        ('Flood/Disaster Forecasting',    'feature_matrix + emdat temporal series', '✅ READY'),
        ('Optimization Engine (OR-Tools)','feature_matrix: warehouse + road features', '✅ READY'),
        ('RL Allocation Agent',           'feature_matrix: vulnerability + logistics', '✅ READY'),
        ('GIS Route Planner',             'road_network graph + warehouse coords', '✅ READY'),
        ('Explainable AI (SHAP)',         'feature_matrix normalized columns', '✅ READY'),
        ('Human-in-the-Loop Dashboard',   'MongoDB collections with indexes', '✅ READY'),
    ]
    for module, input_data, status in modules:
        print(f'  {status}  {module:35s}  [{input_data}]')

    print('\n📁 EXPORT MANIFEST')
    for path in EXPORT_DIR.glob('*'):
        size = path.stat().st_size / 1024
        print(f'  {path.name:50s}  {size:8.1f} KB')

    val_summary = validation_report.get('summary', {})
    print(f'\n🔍 VALIDATION: {val_summary}')

    print('\n' + '='*70)
    print('  Pipeline Complete. ReliefNet data foundation is production-ready.')
    print('='*70)


generate_pipeline_summary(feature_matrix, emdat_df, road_graph, validation_report)

In [ ]:
# ============================================================
# CELL 15.2 — Final Feature Matrix Sample + Schema
# ============================================================

print('\n📊 Final Feature Matrix — Top Districts by Vulnerability:\n')
display_cols = [
    'district', 'state', 'vulnerability_score', 'vulnerability_tier',
    'flood_risk_score', 'avg_road_failure_prob',
    'preferred_delivery_mode', 'disaster_frequency'
]
existing_display = [c for c in display_cols if c in feature_matrix.columns]

display(
    feature_matrix[existing_display]
    .sort_values('vulnerability_score', ascending=False)
    .head(15)
    .reset_index(drop=True)
    .style.background_gradient(
        subset=['vulnerability_score'], cmap='RdYlGn_r'
    ).format({
        'vulnerability_score': '{:.3f}',
        'flood_risk_score':    '{:.3f}',
        'avg_road_failure_prob': '{:.3f}',
    })
)

print(f'\n✅ ReliefNet data engineering pipeline complete.')
print(f'   Feature matrix: {feature_matrix.shape[0]} districts × {feature_matrix.shape[1]} features')
print(f'   All exports saved to: {EXPORT_DIR}')
print(f'   All reports saved to: {REPORT_DIR}')